
# 05 — Robustness and Sensitivity Analysis

Notebook ini menguji apakah kesimpulan utama tetap bertahan ketika keputusan
analitik yang masuk akal diubah.

Analisis mencakup tiga lapisan:

1. **Fungsi produksi lulusan gender–jenjang** dari Notebook 02;
2. **Generalisasi prediktif antarprovinsi** dari Notebook 03;
3. **Komposisi dan aliran keluaran STEM** dari Notebook 04.

## Prinsip

- Analisis sensitivitas tidak digunakan untuk memilih hasil yang paling
  menguntungkan.
- Arah, besar efek, interval ketidakpastian, dan stabilitas antarsampel
  diperiksa secara bersamaan.
- AIC hanya dibandingkan apabila outcome, exposure, dan sampel sama.
- Model dengan exposure mahasiswa aktif menjawab estimand yang berbeda dari
  model dengan exposure mahasiswa terdaftar.
- Seluruh inferensi bersifat agregat, ekologis, dan cross-sectional.

## Analisis yang dilakukan

- ambang denominator alternatif;
- pengecualian pencilan outcome;
- winsorization prediktor;
- spesifikasi kualitas dosen alternatif;
- nonlinearitas;
- covariance HC1, cluster-provinsi, dan cluster-kabupaten/kota;
- efek tetap provinsi;
- *leave-one-province-out*;
- *cluster bootstrap* provinsi;
- stabilitas ranking model prediktif pada kelompok provinsi yang berbeda;
- sensitivitas model komposisi dan aliran STEM.


In [1]:

from __future__ import annotations

import json
import math
import platform
import random
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import patsy
import scipy
import statsmodels
import statsmodels.api as sm
from IPython.display import display
from scipy import stats
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_log_error,
)
from statsmodels.discrete.discrete_model import NegativeBinomial

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

RANDOM_SEED = 42
N_BOOTSTRAP = 100
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option('display.max_columns', 140)
pd.set_option('display.max_rows', 150)
pd.set_option('display.float_format', lambda value: f'{value:,.6f}')

print('Python      :', sys.version.split()[0])
print('Pandas      :', pd.__version__)
print('NumPy       :', np.__version__)
print('SciPy       :', scipy.__version__)
print('Statsmodels :', statsmodels.__version__)
print('Platform    :', platform.platform())
print('Bootstrap   :', N_BOOTSTRAP)


Python      : 3.13.5
Pandas      : 2.2.3
NumPy       : 2.3.5
SciPy       : 1.17.0
Statsmodels : 0.14.6
Platform    : Linux-6.12.13-x86_64-with-glibc2.41
Bootstrap   : 100


In [2]:

# Resolusi folder proyek yang tidak bergantung pada lokasi awal Jupyter.
CURRENT_DIR = Path.cwd().resolve()
WORKBOOK_NAME = 'Dataset_Gabungan_Kemdiktisaintek_2025.xlsx'


def candidate_roots(start: Path) -> list[Path]:
    candidates: list[Path] = []

    def add(path: Path) -> None:
        try:
            resolved = path.resolve()
        except OSError:
            return
        if resolved not in candidates:
            candidates.append(resolved)

    add(start)
    for parent in start.parents:
        add(parent)

    try:
        for child in start.iterdir():
            if child.is_dir():
                add(child)
    except (PermissionError, FileNotFoundError):
        pass

    common_names = [
        'pt_education_research_project',
        'pt_education_research_project_00_01_02_03_04_05',
        'pt_education_research_project_00_01_02_04',
    ]
    for base in list(candidates):
        for name in common_names:
            add(base / name)

    return candidates


PROJECT_ROOT = None
for candidate in candidate_roots(CURRENT_DIR):
    if (candidate / 'data' / 'processed' / WORKBOOK_NAME).exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Folder proyek tidak ditemukan. Ekstrak paket proyek lengkap dan '
        'jalankan Jupyter dari folder proyek.'
    )

OUTPUT_TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
OUTPUT_FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'
OUTPUT_MODEL_DIR = PROJECT_ROOT / 'outputs' / 'models'

for folder in [OUTPUT_TABLE_DIR, OUTPUT_FIGURE_DIR, OUTPUT_MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('CURRENT_DIR :', CURRENT_DIR)
print('PROJECT_ROOT:', PROJECT_ROOT)


CURRENT_DIR : /mnt/data/work_project05/pt_education_research_project
PROJECT_ROOT: /mnt/data/work_project05/pt_education_research_project


In [3]:

# Fungsi mandiri: notebook tidak membutuhkan modul lokal.

def safe_divide(numerator, denominator):
    num = pd.to_numeric(numerator, errors='coerce')
    den = pd.to_numeric(denominator, errors='coerce')
    return num.div(den.where(den.ne(0)))


def add_geo_id(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result['geo_id'] = (
        result['provinsi_pt'].astype(str)
        + ' | '
        + result['kabupaten_pt'].astype(str)
    )
    return result


def save_figure(fig, filename: str) -> Path:
    path = OUTPUT_FIGURE_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return path


def fit_nb2(
    formula: str,
    data: pd.DataFrame,
    *,
    offset_column: str,
    covariance: str = 'cluster_province',
    maxiter: int = 500,
):
    model = NegativeBinomial.from_formula(
        formula,
        data=data,
        offset=np.log(data[offset_column]),
        loglike_method='nb2',
    )

    if covariance == 'cluster_province':
        return model.fit(
            disp=False,
            maxiter=maxiter,
            cov_type='cluster',
            cov_kwds={
                'groups': data['provinsi_pt'],
                'use_correction': True,
            },
        )
    if covariance == 'cluster_district':
        return model.fit(
            disp=False,
            maxiter=maxiter,
            cov_type='cluster',
            cov_kwds={
                'groups': data['geo_id'],
                'use_correction': True,
            },
        )
    if covariance == 'HC1':
        return model.fit(
            disp=False,
            maxiter=maxiter,
            cov_type='HC1',
        )
    if covariance == 'nonrobust':
        return model.fit(disp=False, maxiter=maxiter)

    raise ValueError(f'Covariance tidak dikenal: {covariance}')


def fit_grouped_binomial(
    rhs_formula: str,
    data: pd.DataFrame,
    *,
    covariance: str = 'cluster_province',
):
    design = patsy.dmatrix(rhs_formula, data, return_type='dataframe')
    response = np.column_stack(
        [
            data['jumlah_lulusan_stem'],
            data['jumlah_lulusan_non_stem'],
        ]
    )
    model = sm.GLM(response, design, family=sm.families.Binomial())

    if covariance == 'cluster_province':
        result = model.fit(
            cov_type='cluster',
            cov_kwds={
                'groups': data['provinsi_pt'],
                'use_correction': True,
            },
        )
    elif covariance == 'cluster_district':
        result = model.fit(
            cov_type='cluster',
            cov_kwds={
                'groups': data['geo_id'],
                'use_correction': True,
            },
        )
    elif covariance == 'HC1':
        result = model.fit(cov_type='HC1')
    elif covariance == 'nonrobust':
        result = model.fit()
    else:
        raise ValueError(f'Covariance tidak dikenal: {covariance}')

    return result, design


def extract_ratio_table(result, model_name: str, ratio_name: str) -> pd.DataFrame:
    confidence = result.conf_int()
    table = pd.DataFrame(
        {
            'model': model_name,
            'term': result.params.index,
            'coefficient': result.params.to_numpy(),
            'std_error': result.bse.to_numpy(),
            'z_value': result.tvalues.to_numpy(),
            'p_value': result.pvalues.to_numpy(),
            'ci_lower_coefficient': confidence.iloc[:, 0].to_numpy(),
            'ci_upper_coefficient': confidence.iloc[:, 1].to_numpy(),
        }
    )
    table[ratio_name] = np.exp(table['coefficient'])
    table[f'{ratio_name}_ci_lower'] = np.exp(table['ci_lower_coefficient'])
    table[f'{ratio_name}_ci_upper'] = np.exp(table['ci_upper_coefficient'])
    table['significant_0_05'] = table['p_value'] < 0.05
    return table


def model_row(name: str, result, n_rows: int, metadata: dict) -> dict:
    row = {
        'specification': name,
        'n_rows': int(n_rows),
        'log_likelihood': float(result.llf),
        'aic': float(result.aic),
        'bic': float(getattr(result, 'bic', np.nan)),
        'df_model': float(result.df_model),
        'converged': bool(
            getattr(result, 'mle_retvals', {}).get('converged', True)
        ),
        'covariance': str(result.cov_type),
        'alpha': float(result.params.get('alpha', np.nan)),
    }
    row.update(metadata)
    return row


def term_stability(
    coefficient_table: pd.DataFrame,
    *,
    ratio_column: str,
    comparable_only: bool = True,
) -> pd.DataFrame:
    data = coefficient_table.copy()
    if comparable_only and 'comparable_estimand' in data.columns:
        data = data.loc[data['comparable_estimand']].copy()

    records = []
    for concept, group in data.groupby('concept'):
        ratios = group[ratio_column].dropna()
        if ratios.empty:
            continue
        direction = np.sign(np.log(ratios))
        dominant_direction = 1 if (direction > 0).mean() >= 0.5 else -1
        records.append(
            {
                'concept': concept,
                'n_specifications': len(group),
                'ratio_median': float(ratios.median()),
                'ratio_minimum': float(ratios.min()),
                'ratio_maximum': float(ratios.max()),
                'positive_direction_pct': float((ratios > 1).mean() * 100),
                'negative_direction_pct': float((ratios < 1).mean() * 100),
                'dominant_direction_pct': float(
                    (direction == dominant_direction).mean() * 100
                ),
                'significant_pct': float(
                    group['p_value'].lt(0.05).mean() * 100
                ),
                'ci_excludes_one_pct': float(
                    (
                        (group[f'{ratio_column}_ci_lower'] > 1)
                        | (group[f'{ratio_column}_ci_upper'] < 1)
                    ).mean()
                    * 100
                ),
            }
        )
    return pd.DataFrame(records).sort_values('concept').reset_index(drop=True)


def winsorize_and_standardize(
    frame: pd.DataFrame,
    raw_columns: list[str],
    lower: float = 0.01,
    upper: float = 0.99,
) -> pd.DataFrame:
    result = frame.copy()
    for column in raw_columns:
        low = result[column].quantile(lower)
        high = result[column].quantile(upper)
        clipped = result[column].clip(low, high)
        sd = clipped.std(ddof=0)
        if not np.isfinite(sd) or sd <= 0:
            raise ValueError(f'SD winsorized tidak valid untuk {column}')
        result[f'z_{column}'] = (clipped - clipped.mean()) / sd
    return result


def province_aggregate_predictions(frame: pd.DataFrame) -> pd.DataFrame:
    result = (
        frame.groupby(['scheme', 'model', 'provinsi_pt'], as_index=False)
        .agg(
            observed=('jumlah_lulusan', 'sum'),
            predicted=('predicted_lulusan', 'sum'),
            n_strata=('row_id', 'nunique'),
            registered=('jumlah_terdaftar', 'sum'),
        )
    )
    result['absolute_log_error'] = (
        np.log1p(result['observed']) - np.log1p(result['predicted'])
    ).abs()
    return result


def prediction_metric_row(
    data: pd.DataFrame,
    *,
    scenario: str,
    scheme: str,
    model: str,
) -> dict:
    observed = data['observed'].to_numpy(dtype=float)
    predicted = np.clip(data['predicted'].to_numpy(dtype=float), 0, None)
    return {
        'scenario': scenario,
        'scheme': scheme,
        'model': model,
        'n_provinces': len(data),
        'province_rmsle': float(
            mean_squared_log_error(observed, predicted) ** 0.5
        ),
        'province_mae': float(mean_absolute_error(observed, predicted)),
        'mean_absolute_log_error': float(
            np.mean(np.abs(np.log1p(observed) - np.log1p(predicted)))
        ),
        'median_absolute_log_error': float(
            np.median(np.abs(np.log1p(observed) - np.log1p(predicted)))
        ),
        'spearman_r': float(
            stats.spearmanr(observed, predicted).statistic
        ),
        'predicted_to_observed_ratio': float(
            predicted.sum() / observed.sum()
        ),
    }


def linear_combination_ratio(
    result,
    weights: dict[str, float],
    *,
    label: str,
) -> dict:
    terms = list(result.params.index)
    vector = np.zeros(len(terms), dtype=float)
    for term, weight in weights.items():
        if term in terms:
            vector[terms.index(term)] = weight
    estimate = float(vector @ result.params.to_numpy())
    variance = float(vector @ result.cov_params().to_numpy() @ vector)
    standard_error = math.sqrt(max(variance, 0.0))
    lower = estimate - stats.norm.ppf(0.975) * standard_error
    upper = estimate + stats.norm.ppf(0.975) * standard_error
    z_value = estimate / standard_error if standard_error > 0 else np.nan
    p_value = 2 * stats.norm.sf(abs(z_value)) if np.isfinite(z_value) else np.nan
    return {
        'profile': label,
        'log_ratio': estimate,
        'std_error': standard_error,
        'ratio': math.exp(estimate),
        'ratio_ci_lower': math.exp(lower),
        'ratio_ci_upper': math.exp(upper),
        'p_value': p_value,
    }



## 1. Audit prasyarat dan pemuatan data

Notebook 05 memerlukan hasil Notebook 02, 03, dan 04. Seluruh checklist harus
berstatus `PASS`. Dataset pemodelan yang sudah dikunci digunakan agar analisis
sensitivitas tidak mengubah proses pembersihan data secara diam-diam.


In [4]:

REQUIRED_FILES = {
    'readiness_02': OUTPUT_TABLE_DIR / '02_readiness_checklist.csv',
    'gender_model_data': OUTPUT_TABLE_DIR / '02_modeling_dataset.csv',
    'readiness_03': OUTPUT_TABLE_DIR / '03_readiness_checklist.csv',
    'oof_predictions': OUTPUT_TABLE_DIR / '03_oof_predictions.csv',
    'readiness_04': OUTPUT_TABLE_DIR / '04_readiness_checklist.csv',
    'stem_paired_data': OUTPUT_TABLE_DIR / '04_paired_modeling_dataset.csv',
    'stem_flow_data': OUTPUT_TABLE_DIR / '04_flow_modeling_dataset.csv',
}

prerequisite_rows = []
for label, path in REQUIRED_FILES.items():
    exists = path.exists()
    prerequisite_rows.append(
        {
            'resource': label,
            'path': str(path.relative_to(PROJECT_ROOT)) if exists else str(path),
            'exists': exists,
            'size_bytes': path.stat().st_size if exists else np.nan,
        }
    )

prerequisite_audit = pd.DataFrame(prerequisite_rows)
assert prerequisite_audit['exists'].all(), (
    'File prasyarat belum lengkap:\n'
    + prerequisite_audit.loc[~prerequisite_audit['exists']].to_string(index=False)
)

for label in ['readiness_02', 'readiness_03', 'readiness_04']:
    checklist = pd.read_csv(REQUIRED_FILES[label])
    assert checklist['status'].eq('PASS').all(), f'{label} belum seluruhnya PASS.'

gender_df = add_geo_id(pd.read_csv(REQUIRED_FILES['gender_model_data']))
oof_df = pd.read_csv(REQUIRED_FILES['oof_predictions'])
paired_df = add_geo_id(pd.read_csv(REQUIRED_FILES['stem_paired_data']))
flow_df = add_geo_id(pd.read_csv(REQUIRED_FILES['stem_flow_data']))

# Alternative faculty indicator required by sensitivity specifications.
if 'senior_share' not in paired_df.columns:
    paired_df['senior_share'] = safe_divide(
        paired_df['dosen_senior'], paired_df['dosen_total_jabatan']
    )
for frame in [paired_df, flow_df]:
    sd = frame['senior_share'].std(ddof=0)
    frame['z_senior_share'] = (
        frame['senior_share'] - frame['senior_share'].mean()
    ) / sd

prerequisite_audit.to_csv(
    OUTPUT_TABLE_DIR / '05_prerequisite_audit.csv',
    index=False,
)

display(prerequisite_audit)
print('Gender rows :', f'{len(gender_df):,}')
print('OOF rows    :', f'{len(oof_df):,}')
print('STEM paired :', f'{len(paired_df):,}')
print('STEM flow   :', f'{len(flow_df):,}')


,resource,path,exists,size_bytes
0,readiness_02,outputs/tables/02_readiness_checklist.csv,True,430
1,gender_model_data,outputs/tables/02_modeling_dataset.csv,True,1086494
2,readiness_03,outputs/tables/03_readiness_checklist.csv,True,325
3,oof_predictions,outputs/tables/03_oof_predictions.csv,True,4525633
4,readiness_04,outputs/tables/04_readiness_checklist.csv,True,375
5,stem_paired_data,outputs/tables/04_paired_modeling_dataset.csv,True,285782
6,stem_flow_data,outputs/tables/04_flow_modeling_dataset.csv,True,779817


Gender rows : 2,567
OOF rows    : 35,938
STEM paired : 606
STEM flow   : 1,698



## 2. Robustness model fungsi produksi lulusan gender–jenjang

Model utama Notebook 02 dipertahankan sebagai referensi. Spesifikasi alternatif
mengubah satu keputusan utama pada satu waktu.


In [5]:

GENDER_TERM_FEMALE = 'C(jk, Treatment(reference="L"))[T.P]'
GENDER_BASE_CATEGORICAL = (
    'C(jk, Treatment(reference="L"))'
    ' + C(nm_jenj_didik, Treatment(reference="S1"))'
)
GENDER_MAIN_NUMERIC = (
    'z_active_share + z_mbkm_share + z_log_pt'
    ' + z_pt_high_share + z_prof_share + z_cert_share'
)
GENDER_PRIMARY_FORMULA = (
    'jumlah_lulusan ~ '
    + GENDER_BASE_CATEGORICAL
    + ' + '
    + GENDER_MAIN_NUMERIC
)

# Winsorized sample uses the same variable names and therefore the same formula.
GENDER_RAW_FOR_WINSOR = [
    'active_share',
    'mbkm_share',
    'log_pt',
    'pt_high_share',
    'prof_share',
    'cert_share',
]
gender_winsor = winsorize_and_standardize(
    gender_df,
    GENDER_RAW_FOR_WINSOR,
    lower=0.01,
    upper=0.99,
)

outcome_cutoff_99 = gender_df['jumlah_lulusan'].quantile(0.99)
active_offset_df = gender_df.loc[gender_df['jumlah_aktif_total'] > 0].copy()

GENDER_SPECIFICATIONS = [
    {
        'name': 'G00_primary',
        'data': gender_df,
        'formula': GENDER_PRIMARY_FORMULA,
        'offset': 'jumlah_terdaftar',
        'covariance': 'cluster_province',
        'sample_rule': 'complete-case Notebook 02',
        'estimand': 'per registered-student stock',
        'comparable_estimand': True,
    },
    *[
        {
            'name': f'G_denominator_ge_{threshold}',
            'data': gender_df.loc[gender_df['jumlah_terdaftar'] >= threshold].copy(),
            'formula': GENDER_PRIMARY_FORMULA,
            'offset': 'jumlah_terdaftar',
            'covariance': 'cluster_province',
            'sample_rule': f'jumlah_terdaftar >= {threshold}',
            'estimand': 'per registered-student stock',
            'comparable_estimand': True,
        }
        for threshold in [25, 50, 100, 250, 500]
    ],
    {
        'name': 'G_exclude_top_1pct_outcome',
        'data': gender_df.loc[
            gender_df['jumlah_lulusan'] <= outcome_cutoff_99
        ].copy(),
        'formula': GENDER_PRIMARY_FORMULA,
        'offset': 'jumlah_terdaftar',
        'covariance': 'cluster_province',
        'sample_rule': 'exclude outcome above percentile 99',
        'estimand': 'per registered-student stock',
        'comparable_estimand': True,
    },
    {
        'name': 'G_winsorize_predictors_1_99',
        'data': gender_winsor,
        'formula': GENDER_PRIMARY_FORMULA,
        'offset': 'jumlah_terdaftar',
        'covariance': 'cluster_province',
        'sample_rule': 'winsorize continuous predictors at 1st/99th percentile',
        'estimand': 'per registered-student stock',
        'comparable_estimand': True,
    },
    {
        'name': 'G_replace_professor_with_senior',
        'data': gender_df,
        'formula': GENDER_PRIMARY_FORMULA.replace(
            'z_prof_share', 'z_senior_share'
        ),
        'offset': 'jumlah_terdaftar',
        'covariance': 'cluster_province',
        'sample_rule': 'senior faculty share replaces professor share',
        'estimand': 'per registered-student stock',
        'comparable_estimand': True,
    },
    {
        'name': 'G_without_faculty_block',
        'data': gender_df,
        'formula': (
            'jumlah_lulusan ~ '
            + GENDER_BASE_CATEGORICAL
            + ' + z_active_share + z_mbkm_share + z_log_pt + z_pt_high_share'
        ),
        'offset': 'jumlah_terdaftar',
        'covariance': 'cluster_province',
        'sample_rule': 'faculty quality block removed',
        'estimand': 'per registered-student stock',
        'comparable_estimand': True,
    },
    {
        'name': 'G_quadratic_student_participation',
        'data': gender_df,
        'formula': (
            GENDER_PRIMARY_FORMULA
            + ' + I(z_active_share ** 2) + I(z_mbkm_share ** 2)'
        ),
        'offset': 'jumlah_terdaftar',
        'covariance': 'cluster_province',
        'sample_rule': 'quadratic active and MBKM shares',
        'estimand': 'per registered-student stock',
        'comparable_estimand': True,
    },
    {
        'name': 'G_province_fixed_effects',
        'data': gender_df,
        'formula': GENDER_PRIMARY_FORMULA + ' + C(provinsi_pt)',
        'offset': 'jumlah_terdaftar',
        'covariance': 'cluster_province',
        'sample_rule': 'province fixed effects',
        'estimand': 'within-province conditional association',
        'comparable_estimand': True,
    },
    {
        'name': 'G_cluster_district',
        'data': gender_df,
        'formula': GENDER_PRIMARY_FORMULA,
        'offset': 'jumlah_terdaftar',
        'covariance': 'cluster_district',
        'sample_rule': 'district-cluster covariance',
        'estimand': 'per registered-student stock',
        'comparable_estimand': True,
    },
    {
        'name': 'G_HC1_covariance',
        'data': gender_df,
        'formula': GENDER_PRIMARY_FORMULA,
        'offset': 'jumlah_terdaftar',
        'covariance': 'HC1',
        'sample_rule': 'HC1 covariance',
        'estimand': 'per registered-student stock',
        'comparable_estimand': True,
    },
    {
        'name': 'G_active_student_offset',
        'data': active_offset_df,
        'formula': (
            'jumlah_lulusan ~ '
            + GENDER_BASE_CATEGORICAL
            + ' + z_mbkm_share + z_log_pt + z_pt_high_share'
            + ' + z_prof_share + z_cert_share'
        ),
        'offset': 'jumlah_aktif_total',
        'covariance': 'cluster_province',
        'sample_rule': 'active-student stock used as exposure',
        'estimand': 'per active-student stock',
        'comparable_estimand': False,
    },
]

GENDER_CONCEPT_MAP = {
    GENDER_TERM_FEMALE: 'female_vs_male',
    'z_active_share': 'active_share_per_sd',
    'z_mbkm_share': 'mbkm_share_per_sd',
    'z_log_pt': 'log_number_of_institutions_per_sd',
    'z_pt_high_share': 'high_accreditation_institution_share_per_sd',
    'z_prof_share': 'professor_share_per_sd',
    'z_senior_share': 'senior_faculty_share_per_sd',
    'z_cert_share': 'certified_faculty_share_per_sd',
}

gender_results = {}
gender_model_rows = []
gender_coefficient_rows = []

for specification in GENDER_SPECIFICATIONS:
    name = specification['name']
    data = specification['data']
    print('Fitting', name, 'n=', len(data))
    result = fit_nb2(
        specification['formula'],
        data,
        offset_column=specification['offset'],
        covariance=specification['covariance'],
    )
    gender_results[name] = result

    metadata = {
        'sample_rule': specification['sample_rule'],
        'offset_column': specification['offset'],
        'estimand': specification['estimand'],
        'comparable_estimand': specification['comparable_estimand'],
    }
    gender_model_rows.append(model_row(name, result, len(data), metadata))

    table = extract_ratio_table(result, name, 'irr')
    table = table.loc[table['term'].isin(GENDER_CONCEPT_MAP)].copy()
    table['concept'] = table['term'].map(GENDER_CONCEPT_MAP)
    for key, value in metadata.items():
        table[key] = value
    gender_coefficient_rows.append(table)

gender_specification_grid = pd.DataFrame(gender_model_rows)
gender_coefficients = pd.concat(gender_coefficient_rows, ignore_index=True)
gender_stability = term_stability(
    gender_coefficients,
    ratio_column='irr',
    comparable_only=True,
)

gender_specification_grid.to_csv(
    OUTPUT_TABLE_DIR / '05_gender_specification_grid.csv',
    index=False,
)
gender_coefficients.to_csv(
    OUTPUT_TABLE_DIR / '05_gender_specification_coefficients.csv',
    index=False,
)
gender_stability.to_csv(
    OUTPUT_TABLE_DIR / '05_gender_term_stability.csv',
    index=False,
)

display(gender_specification_grid)
display(gender_stability)
assert gender_specification_grid['converged'].all()


Fitting G00_primary n= 2567


Fitting G_denominator_ge_25 n= 2373


Fitting G_denominator_ge_50 n= 2198


Fitting G_denominator_ge_100 n= 1980


Fitting G_denominator_ge_250 n= 1591


...

Fitting G_province_fixed_effects n= 2567


Fitting G_cluster_district n= 2567
Fitting G_HC1_covariance n= 2567


Fitting G_active_student_offset n= 2567


,specification,n_rows,log_likelihood,aic,bic,df_model,converged,covariance,alpha,sample_rule,offset_column,estimand,comparable_estimand
0,G00_primary,2567,"-14,364.316119","28,766.632238","28,877.791609",17.000000,True,cluster,0.207689,complete-case Notebook 02,jumlah_terdaftar,per registered-student stock,True
1,G_denominator_ge_25,2373,"-13,835.789303","27,709.578606","27,819.244901",17.000000,True,cluster,0.204893,jumlah_terdaftar >= 25,jumlah_terdaftar,per registered-student stock,True
2,G_denominator_ge_50,2198,"-13,245.143491","26,528.286983","26,636.497742",17.000000,True,cluster,0.199805,jumlah_terdaftar >= 50,jumlah_terdaftar,per registered-student stock,True
3,G_denominator_ge_100,1980,"-12,324.041759","24,686.083517","24,792.309708",17.000000,True,cluster,0.181276,jumlah_terdaftar >= 100,jumlah_terdaftar,per registered-student stock,True
4,G_denominator_ge_250,1591,"-10,416.407728","20,870.815456","20,972.885699",17.000000,True,cluster,0.148000,jumlah_terdaftar >= 250,jumlah_terdaftar,per registered-student stock,True
5,G_denominator_ge_500,1275,"-8,675.661787","17,383.323574","17,465.734797",14.000000,True,cluster,0.115344,jumlah_terdaftar >= 500,jumlah_terdaftar,per registered-student stock,True
6,G_exclude_top_1pct_outcome,2541,"-14,089.279576","28,216.559152","28,327.525099",17.000000,True,cluster,0.208986,exclude outcome above percentile 99,jumlah_terdaftar,per registered-student stock,True
7,G_winsorize_predictors_1_99,2567,"-14,362.815760","28,763.631520","28,874.790891",17.000000,True,cluster,0.207314,winsorize continuous predictors at 1st/99th pe...,jumlah_terdaftar,per registered-student stock,True
8,G_replace_professor_with_senior,2567,"-14,363.191747","28,764.383494","28,875.542864",17.000000,True,cluster,0.207472,senior faculty share replaces professor share,jumlah_terdaftar,per registered-student stock,True
9,G_without_faculty_block,2567,"-14,366.323629","28,766.647258","28,866.105642",15.000000,True,cluster,0.208180,faculty quality block removed,jumlah_terdaftar,per registered-student stock,True


,concept,n_specifications,ratio_median,ratio_minimum,ratio_maximum,positive_direction_pct,negative_direction_pct,dominant_direction_pct,significant_pct,ci_excludes_one_pct
0,active_share_per_sd,14,1.192719,1.168060,1.203541,100.000000,0.000000,100.000000,100.000000,100.000000
1,certified_faculty_share_per_sd,13,1.010805,1.001214,1.015703,100.000000,0.000000,100.000000,0.000000,0.000000
2,female_vs_male,14,1.094230,1.092200,1.156690,100.000000,0.000000,100.000000,100.000000,100.000000
3,high_accreditation_institution_share_per_sd,14,1.018831,1.008516,1.033207,100.000000,0.000000,100.000000,0.000000,0.000000
4,log_number_of_institutions_per_sd,14,0.979386,0.977843,1.031659,14.285714,85.714286,85.714286,7.142857,7.142857
5,mbkm_share_per_sd,14,0.990791,0.988074,0.999933,0.000000,100.000000,100.000000,0.000000,0.000000
6,professor_share_per_sd,12,1.019415,0.998577,1.022174,91.666667,8.333333,91.666667,0.000000,0.000000
7,senior_faculty_share_per_sd,1,1.032671,1.032671,1.032671,100.000000,0.000000,100.000000,0.000000,0.000000


In [6]:

# Visualisasi IRR perempuan dan proporsi mahasiswa aktif pada seluruh
# spesifikasi yang memiliki estimand comparable.
for concept, filename, title in [
    (
        'female_vs_male',
        '05_01_gender_female_irr_across_specifications.png',
        'IRR Perempuan vs Laki-laki pada Spesifikasi Alternatif',
    ),
    (
        'active_share_per_sd',
        '05_02_gender_active_share_irr_across_specifications.png',
        'IRR Proporsi Mahasiswa Aktif pada Spesifikasi Alternatif',
    ),
]:
    plot_data = gender_coefficients.loc[
        gender_coefficients['concept'].eq(concept)
        & gender_coefficients['comparable_estimand']
    ].copy()
    plot_data = plot_data.sort_values('irr')

    fig_height = max(6, 0.38 * len(plot_data))
    fig, ax = plt.subplots(figsize=(10, fig_height))
    positions = np.arange(len(plot_data))
    errors = np.vstack(
        [
            plot_data['irr'] - plot_data['irr_ci_lower'],
            plot_data['irr_ci_upper'] - plot_data['irr'],
        ]
    )
    ax.errorbar(
        plot_data['irr'],
        positions,
        xerr=errors,
        fmt='o',
        capsize=3,
    )
    ax.axvline(1, linestyle='--', linewidth=1)
    ax.set_yticks(positions)
    ax.set_yticklabels(plot_data['model'])
    ax.set_xscale('log')
    ax.set_xlabel('Incidence Rate Ratio (skala log)')
    ax.set_title(title)
    ax.grid(axis='x', alpha=0.25)
    save_figure(fig, filename)



## 3. Leave-one-province-out untuk model gender–jenjang

Setiap provinsi dikeluarkan satu per satu. Analisis ini mendeteksi apakah satu
provinsi tertentu mengendalikan arah atau besar kesimpulan utama.


In [7]:

GENDER_LOO_TERMS = {
    GENDER_TERM_FEMALE: 'female_vs_male',
    'z_active_share': 'active_share_per_sd',
    'z_mbkm_share': 'mbkm_share_per_sd',
    'z_log_pt': 'log_number_of_institutions_per_sd',
    'z_pt_high_share': 'high_accreditation_institution_share_per_sd',
    'z_prof_share': 'professor_share_per_sd',
    'z_cert_share': 'certified_faculty_share_per_sd',
}

primary_gender_result = gender_results['G00_primary']
full_log_irr = {
    concept: float(primary_gender_result.params[term])
    for term, concept in GENDER_LOO_TERMS.items()
}

gender_loo_rows = []
for province in sorted(gender_df['provinsi_pt'].unique()):
    subset = gender_df.loc[gender_df['provinsi_pt'].ne(province)].copy()
    result = fit_nb2(
        GENDER_PRIMARY_FORMULA,
        subset,
        offset_column='jumlah_terdaftar',
        covariance='cluster_province',
    )
    for term, concept in GENDER_LOO_TERMS.items():
        coefficient = float(result.params[term])
        confidence = result.conf_int().loc[term]
        gender_loo_rows.append(
            {
                'excluded_province': province,
                'concept': concept,
                'term': term,
                'coefficient': coefficient,
                'irr': math.exp(coefficient),
                'irr_ci_lower': math.exp(float(confidence.iloc[0])),
                'irr_ci_upper': math.exp(float(confidence.iloc[1])),
                'p_value': float(result.pvalues[term]),
                'log_irr_deviation_from_full': (
                    coefficient - full_log_irr[concept]
                ),
                'absolute_log_irr_deviation': abs(
                    coefficient - full_log_irr[concept]
                ),
                'alpha': float(result.params['alpha']),
                'converged': bool(
                    result.mle_retvals.get('converged', False)
                ),
            }
        )

gender_loo = pd.DataFrame(gender_loo_rows)
gender_loo_summary = (
    gender_loo.groupby('concept', as_index=False)
    .agg(
        n_exclusions=('excluded_province', 'nunique'),
        irr_median=('irr', 'median'),
        irr_minimum=('irr', 'min'),
        irr_maximum=('irr', 'max'),
        positive_direction_pct=('irr', lambda x: (x > 1).mean() * 100),
        significant_pct=('p_value', lambda x: (x < 0.05).mean() * 100),
        maximum_absolute_log_irr_deviation=(
            'absolute_log_irr_deviation', 'max'
        ),
    )
)

gender_influential_provinces = (
    gender_loo.groupby('excluded_province', as_index=False)
    .agg(
        maximum_absolute_log_irr_deviation=(
            'absolute_log_irr_deviation', 'max'
        ),
        mean_absolute_log_irr_deviation=(
            'absolute_log_irr_deviation', 'mean'
        ),
    )
    .sort_values('maximum_absolute_log_irr_deviation', ascending=False)
)

gender_loo.to_csv(
    OUTPUT_TABLE_DIR / '05_gender_leave_one_province_out.csv',
    index=False,
)
gender_loo_summary.to_csv(
    OUTPUT_TABLE_DIR / '05_gender_leave_one_province_out_summary.csv',
    index=False,
)
gender_influential_provinces.to_csv(
    OUTPUT_TABLE_DIR / '05_gender_influential_provinces.csv',
    index=False,
)

display(gender_loo_summary)
display(gender_influential_provinces.head(10))
assert gender_loo['converged'].all()
assert gender_loo['excluded_province'].nunique() == gender_df['provinsi_pt'].nunique()


,concept,n_exclusions,irr_median,irr_minimum,irr_maximum,positive_direction_pct,significant_pct,maximum_absolute_log_irr_deviation
0,active_share_per_sd,34,1.195338,1.179420,1.203035,100.000000,100.000000,0.012473
1,certified_faculty_share_per_sd,34,1.010912,0.998856,1.016624,97.058824,0.000000,0.011892
2,female_vs_male,34,1.094192,1.085039,1.100462,100.000000,100.000000,0.008435
3,high_accreditation_institution_share_per_sd,34,1.018428,1.008350,1.054070,100.000000,0.000000,0.033912
4,log_number_of_institutions_per_sd,34,0.977802,0.973106,0.993152,0.000000,0.000000,0.014854
5,mbkm_share_per_sd,34,0.990669,0.983542,0.995645,0.000000,0.000000,0.006996
6,professor_share_per_sd,34,1.022082,1.005236,1.035787,100.000000,0.000000,0.016285


,excluded_province,maximum_absolute_log_irr_deviation,mean_absolute_log_irr_deviation
8,Prov. Jambi,0.033912,0.007982
30,Prov. Sulawesi Utara,0.016285,0.009831
9,Prov. Jawa Barat,0.013653,0.003788
20,Prov. Maluku Utara,0.011892,0.006467
24,Prov. Papua Barat,0.009468,0.003138
0,Prov. Aceh,0.008889,0.002292
33,Prov. Sumatera Utara,0.008435,0.004170
10,Prov. Jawa Tengah,0.006996,0.004413
1,Prov. Bali,0.006817,0.001904
23,Prov. Papua,0.006167,0.001824


In [8]:

fig, ax = plt.subplots(figsize=(9, 8))
plot_data = gender_influential_provinces.sort_values(
    'maximum_absolute_log_irr_deviation'
)
ax.barh(
    plot_data['excluded_province'],
    plot_data['maximum_absolute_log_irr_deviation'],
)
ax.set_xlabel('Maksimum perubahan absolut log(IRR)')
ax.set_ylabel('')
ax.set_title('Pengaruh Pengeluaran Satu Provinsi terhadap Koefisien Utama')
ax.grid(axis='x', alpha=0.25)
save_figure(fig, '05_03_gender_influential_provinces.png')


PosixPath('/mnt/data/work_project05/pt_education_research_project/outputs/figures/05_03_gender_influential_provinces.png')


## 4. Cluster bootstrap provinsi

Provinsi diambil ulang dengan pengembalian. Seluruh strata dalam provinsi yang
terpilih ikut masuk sebagai satu cluster. Interval percentile bootstrap memberi
ukuran ketidakpastian yang tidak hanya mengandalkan aproksimasi normal.


In [9]:

rng = np.random.default_rng(RANDOM_SEED)
provinces = np.array(sorted(gender_df['provinsi_pt'].unique()))
bootstrap_rows = []
bootstrap_failures = []

for replicate in range(1, N_BOOTSTRAP + 1):
    sampled_provinces = rng.choice(
        provinces,
        size=len(provinces),
        replace=True,
    )
    parts = []
    for draw_id, province in enumerate(sampled_provinces):
        part = gender_df.loc[
            gender_df['provinsi_pt'].eq(province)
        ].copy()
        part['_bootstrap_draw_id'] = draw_id
        parts.append(part)
    bootstrap_data = pd.concat(parts, ignore_index=True)

    try:
        result = fit_nb2(
            GENDER_PRIMARY_FORMULA,
            bootstrap_data,
            offset_column='jumlah_terdaftar',
            covariance='nonrobust',
            maxiter=350,
        )
        if not result.mle_retvals.get('converged', False):
            raise RuntimeError('model did not converge')
        for term, concept in GENDER_LOO_TERMS.items():
            bootstrap_rows.append(
                {
                    'replicate': replicate,
                    'concept': concept,
                    'term': term,
                    'coefficient': float(result.params[term]),
                    'irr': math.exp(float(result.params[term])),
                    'alpha': float(result.params['alpha']),
                    'n_unique_provinces': int(len(set(sampled_provinces))),
                }
            )
    except Exception as error:
        bootstrap_failures.append(
            {
                'replicate': replicate,
                'error': f'{type(error).__name__}: {error}',
            }
        )

bootstrap_estimates = pd.DataFrame(bootstrap_rows)
bootstrap_failure_table = pd.DataFrame(bootstrap_failures)
bootstrap_successes = bootstrap_estimates['replicate'].nunique()

bootstrap_summary_rows = []
for concept, group in bootstrap_estimates.groupby('concept'):
    full_estimate = math.exp(full_log_irr[concept])
    bootstrap_summary_rows.append(
        {
            'concept': concept,
            'successful_replicates': group['replicate'].nunique(),
            'full_sample_irr': full_estimate,
            'bootstrap_irr_mean': float(group['irr'].mean()),
            'bootstrap_irr_median': float(group['irr'].median()),
            'bootstrap_ci_lower_2_5': float(group['irr'].quantile(0.025)),
            'bootstrap_ci_upper_97_5': float(group['irr'].quantile(0.975)),
            'positive_direction_pct': float((group['irr'] > 1).mean() * 100),
        }
    )
bootstrap_summary = pd.DataFrame(bootstrap_summary_rows)

bootstrap_estimates.to_csv(
    OUTPUT_TABLE_DIR / '05_gender_cluster_bootstrap_estimates.csv',
    index=False,
)
bootstrap_summary.to_csv(
    OUTPUT_TABLE_DIR / '05_gender_cluster_bootstrap_summary.csv',
    index=False,
)
bootstrap_failure_table.to_csv(
    OUTPUT_TABLE_DIR / '05_gender_cluster_bootstrap_failures.csv',
    index=False,
)

display(bootstrap_summary)
print('Successful bootstrap replicates:', bootstrap_successes)
assert bootstrap_successes >= int(0.9 * N_BOOTSTRAP)


,concept,successful_replicates,full_sample_irr,bootstrap_irr_mean,bootstrap_irr_median,bootstrap_ci_lower_2_5,bootstrap_ci_upper_97_5,positive_direction_pct
0,active_share_per_sd,100,1.194223,1.186124,1.183733,1.124681,1.259614,100.000000
1,certified_faculty_share_per_sd,100,1.010805,1.007141,1.007440,0.978009,1.040690,66.000000
2,female_vs_male,100,1.094230,1.097071,1.097495,1.067238,1.120535,100.000000
3,high_accreditation_institution_share_per_sd,100,1.018924,1.027385,1.018720,0.977634,1.146234,77.000000
4,log_number_of_institutions_per_sd,100,0.978508,0.981558,0.984910,0.944238,1.008369,10.000000
5,mbkm_share_per_sd,100,0.990447,0.993848,0.993402,0.968554,1.012550,30.000000
6,professor_share_per_sd,100,1.021741,1.019965,1.018071,0.973816,1.077663,76.000000


Successful bootstrap replicates: 100


In [10]:

bootstrap_plot_concepts = [
    'female_vs_male',
    'active_share_per_sd',
    'mbkm_share_per_sd',
    'high_accreditation_institution_share_per_sd',
]
plot_groups = [
    bootstrap_estimates.loc[
        bootstrap_estimates['concept'].eq(concept), 'irr'
    ].to_numpy()
    for concept in bootstrap_plot_concepts
]

fig, ax = plt.subplots(figsize=(10, 6))
ax.boxplot(plot_groups, labels=bootstrap_plot_concepts, showfliers=False)
ax.axhline(1, linestyle='--', linewidth=1)
ax.set_yscale('log')
ax.set_ylabel('Bootstrap IRR (skala log)')
ax.set_title('Distribusi Cluster-Bootstrap IRR Provinsi')
ax.tick_params(axis='x', rotation=25)
ax.grid(axis='y', alpha=0.25)
save_figure(fig, '05_04_gender_cluster_bootstrap_irr.png')


/tmp/ipykernel_4866/177290494.py:15: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(plot_groups, labels=bootstrap_plot_concepts, showfliers=False)


PosixPath('/mnt/data/work_project05/pt_education_research_project/outputs/figures/05_04_gender_cluster_bootstrap_irr.png')


## 5. Robustness generalisasi prediktif Notebook 03

Model dinilai pada agregat provinsi agar setiap provinsi mendapat bobot yang
sama. Ranking diuji pada seluruh provinsi, provinsi berukuran kecil/besar, dan
setelah provinsi dengan volume terbesar dikeluarkan.


In [11]:

province_predictions = province_aggregate_predictions(oof_df)

group_province = province_predictions.loc[
    province_predictions['scheme'].eq('GroupKFold_Province')
].copy()
province_volume = (
    group_province.groupby('provinsi_pt')['observed'].first().sort_values()
)
median_volume = float(province_volume.median())
top_three = set(province_volume.nlargest(3).index)
top_decile_count = max(1, math.ceil(len(province_volume) * 0.10))
top_decile = set(province_volume.nlargest(top_decile_count).index)

SCENARIO_PROVINCES = {
    'all_provinces': set(province_volume.index),
    'exclude_top_3_volume': set(province_volume.index) - top_three,
    'exclude_top_10pct_volume': set(province_volume.index) - top_decile,
    'low_volume_half': set(province_volume.loc[province_volume <= median_volume].index),
    'high_volume_half': set(province_volume.loc[province_volume > median_volume].index),
}

predictive_scenario_rows = []
for scenario, included_provinces in SCENARIO_PROVINCES.items():
    scenario_data = group_province.loc[
        group_province['provinsi_pt'].isin(included_provinces)
    ]
    for model, group in scenario_data.groupby('model'):
        predictive_scenario_rows.append(
            prediction_metric_row(
                group,
                scenario=scenario,
                scheme='GroupKFold_Province',
                model=model,
            )
        )

predictive_scenarios = pd.DataFrame(predictive_scenario_rows)
predictive_scenarios['rmsle_rank'] = predictive_scenarios.groupby(
    'scenario'
)['province_rmsle'].rank(method='min')
predictive_scenarios['mae_rank'] = predictive_scenarios.groupby(
    'scenario'
)['province_mae'].rank(method='min')

rank_stability = (
    predictive_scenarios.groupby('model', as_index=False)
    .agg(
        scenarios=('scenario', 'nunique'),
        mean_rmsle_rank=('rmsle_rank', 'mean'),
        median_rmsle_rank=('rmsle_rank', 'median'),
        worst_rmsle_rank=('rmsle_rank', 'max'),
        mean_mae_rank=('mae_rank', 'mean'),
        rmsle_wins=('rmsle_rank', lambda x: int((x == 1).sum())),
        rmsle_top_3=('rmsle_rank', lambda x: int((x <= 3).sum())),
        rmsle_mean=('province_rmsle', 'mean'),
        rmsle_maximum=('province_rmsle', 'max'),
    )
    .sort_values(['mean_rmsle_rank', 'rmsle_mean'])
)

predictive_scenarios.to_csv(
    OUTPUT_TABLE_DIR / '05_predictive_scenario_metrics.csv',
    index=False,
)
rank_stability.to_csv(
    OUTPUT_TABLE_DIR / '05_predictive_rank_stability.csv',
    index=False,
)

display(
    predictive_scenarios.sort_values(['scenario', 'province_rmsle'])
)
display(rank_stability)


,scenario,scheme,model,n_provinces,province_rmsle,province_mae,mean_absolute_log_error,median_absolute_log_error,spearman_r,predicted_to_observed_ratio,rmsle_rank,mae_rank
6,all_provinces,GroupKFold_Province,TweedieRate,34,0.225527,"11,260.176799",0.181415,0.124056,0.985027,1.034721,1.000000,3.000000
4,all_provinces,GroupKFold_Province,PoissonRate,34,0.226900,"11,402.150580",0.180092,0.114521,0.984721,1.036140,2.000000,4.000000
1,all_provinces,GroupKFold_Province,ExtraTreesLogCount,34,0.238169,"9,260.136395",0.163147,0.107526,0.983804,0.904854,3.000000,1.000000
2,all_provinces,GroupKFold_Province,HistGBPoissonRate,34,0.247037,"10,985.622955",0.208796,0.202261,0.971276,1.054821,4.000000,2.000000
5,all_provinces,GroupKFold_Province,RandomForestLogCount,34,0.247614,"11,819.489390",0.185123,0.131609,0.986249,0.880812,5.000000,5.000000
3,all_provinces,GroupKFold_Province,NB2_Offset,34,0.250585,"11,990.647449",0.212439,0.181648,0.982582,1.095490,6.000000,6.000000
0,all_provinces,GroupKFold_Province,ExposureRateBaseline,34,0.343849,"23,252.607301",0.239012,0.175810,0.962108,1.022100,7.000000,7.000000
20,exclude_top_10pct_volume,GroupKFold_Province,TweedieRate,30,0.238698,"10,211.915784",0.196744,0.140501,0.978643,1.081645,1.000000,4.000000
18,exclude_top_10pct_volume,GroupKFold_Province,PoissonRate,30,0.240317,"10,537.643572",0.195954,0.160052,0.978198,1.087545,2.000000,5.000000
15,exclude_top_10pct_volume,GroupKFold_Province,ExtraTreesLogCount,30,0.250710,"7,241.534252",0.174803,0.121437,0.976418,0.894310,3.000000,1.000000


,model,scenarios,mean_rmsle_rank,median_rmsle_rank,worst_rmsle_rank,mean_mae_rank,rmsle_wins,rmsle_top_3,rmsle_mean,rmsle_maximum
6,TweedieRate,5,1.600000,1.000000,3.000000,3.400000,3,5,0.230064,0.238698
4,PoissonRate,5,2.200000,2.000000,4.000000,4.400000,1,4,0.231540,0.240317
1,ExtraTreesLogCount,5,3.400000,3.000000,5.000000,1.000000,0,4,0.242357,0.250710
2,HistGBPoissonRate,5,4.200000,5.000000,5.000000,3.200000,0,1,0.251055,0.281527
5,RandomForestLogCount,5,4.600000,4.000000,6.000000,3.400000,0,0,0.251559,0.259278
3,NB2_Offset,5,5.000000,6.000000,6.000000,6.000000,1,1,0.254325,0.289334
0,ExposureRateBaseline,5,7.000000,7.000000,7.000000,6.600000,0,0,0.349101,0.373217


In [12]:

# Generalization gap sensitivity at alternative minimum exposure thresholds.
threshold_rows = []
for threshold in [1, 25, 50, 100, 250, 500]:
    threshold_data = oof_df.loc[
        oof_df['jumlah_terdaftar'] >= threshold
    ].copy()
    for (scheme, model), group in threshold_data.groupby(['scheme', 'model']):
        observed = group['jumlah_lulusan'].to_numpy(dtype=float)
        predicted = np.clip(
            group['predicted_lulusan'].to_numpy(dtype=float),
            0,
            None,
        )
        threshold_rows.append(
            {
                'minimum_registered': threshold,
                'scheme': scheme,
                'model': model,
                'n_strata': len(group),
                'stratum_rmsle': float(
                    mean_squared_log_error(observed, predicted) ** 0.5
                ),
            }
        )

threshold_metrics = pd.DataFrame(threshold_rows)
gap_table = (
    threshold_metrics.pivot_table(
        index=['minimum_registered', 'model'],
        columns='scheme',
        values='stratum_rmsle',
    )
    .reset_index()
)
gap_table['group_minus_random_rmsle'] = (
    gap_table['GroupKFold_Province'] - gap_table['RandomKFold']
)
gap_table['group_to_random_ratio'] = safe_divide(
    gap_table['GroupKFold_Province'],
    gap_table['RandomKFold'],
)

gap_table.to_csv(
    OUTPUT_TABLE_DIR / '05_predictive_generalization_gap_thresholds.csv',
    index=False,
)
display(gap_table)


scheme,minimum_registered,model,GroupKFold_Province,RandomKFold,group_minus_random_rmsle,group_to_random_ratio
0,1,ExposureRateBaseline,0.825102,0.824729,0.000373,1.000453
1,1,ExtraTreesLogCount,0.545638,0.535554,0.010085,1.018830
2,1,HistGBPoissonRate,0.557271,0.506081,0.051190,1.101150
3,1,NB2_Offset,0.536459,0.531821,0.004638,1.008721
4,1,PoissonRate,0.528987,0.522759,0.006228,1.011913
5,1,RandomForestLogCount,0.551233,0.540583,0.010651,1.019703
6,1,TweedieRate,0.542681,0.540654,0.002027,1.003749
7,25,ExposureRateBaseline,0.832459,0.832216,0.000243,1.000292
8,25,ExtraTreesLogCount,0.546569,0.535797,0.010771,1.020104
9,25,HistGBPoissonRate,0.550192,0.495072,0.055121,1.111339


In [13]:

fig, ax = plt.subplots(figsize=(10, 6))
plot_rank = rank_stability.sort_values('mean_rmsle_rank', ascending=False)
ax.barh(plot_rank['model'], plot_rank['mean_rmsle_rank'])
ax.set_xlabel('Rata-rata ranking RMSLE provinsi (lebih rendah lebih baik)')
ax.set_ylabel('')
ax.set_title('Stabilitas Ranking Model pada Skenario Provinsi')
ax.grid(axis='x', alpha=0.25)
save_figure(fig, '05_05_predictive_rank_stability.png')

fig, ax = plt.subplots(figsize=(10, 6))
for model, group in gap_table.groupby('model'):
    group = group.sort_values('minimum_registered')
    ax.plot(
        group['minimum_registered'],
        group['group_minus_random_rmsle'],
        marker='o',
        label=model,
    )
ax.axhline(0, linestyle='--', linewidth=1)
ax.set_xscale('log')
ax.set_xlabel('Minimum mahasiswa terdaftar per strata')
ax.set_ylabel('RMSLE GroupCV minus RandomCV')
ax.set_title('Sensitivitas Generalization Gap terhadap Denominator')
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.25)
save_figure(fig, '05_06_predictive_generalization_gap_thresholds.png')


PosixPath('/mnt/data/work_project05/pt_education_research_project/outputs/figures/05_06_predictive_generalization_gap_thresholds.png')


## 6. Sensitivitas model komposisi lulusan STEM

Outcome grouped-binomial membandingkan lulusan STEM dengan non-STEM dalam
strata wilayah×jenjang yang sama. Koefisien utama adalah komposisi mahasiswa
baru STEM.


In [14]:

BIN_PRIMARY_RHS = (
    'C(nm_jenj_didik, Treatment(reference="S1"))'
    ' + z_stem_new_share'
    ' + z_stem_new_prodi_high_share'
    ' + z_pt_high_share'
    ' + z_doctor_stem_share'
    ' + z_log_doctor_total'
    ' + z_prof_share'
    ' + z_cert_share'
)
BIN_NO_FACULTY_RHS = (
    'C(nm_jenj_didik, Treatment(reference="S1"))'
    ' + z_stem_new_share'
    ' + z_stem_new_prodi_high_share'
    ' + z_pt_high_share'
    ' + z_doctor_stem_share'
    ' + z_log_doctor_total'
)
BIN_ENTRY_TERM = 'z_stem_new_share'

bin_outcome_cutoff = paired_df['total_lulusan'].quantile(0.99)
BIN_SPECIFICATIONS = [
    {
        'name': 'B00_primary',
        'data': paired_df,
        'rhs': BIN_PRIMARY_RHS,
        'covariance': 'cluster_province',
        'sample_rule': 'Notebook 04 paired complete-case',
    },
    *[
        {
            'name': f'B_total_graduates_ge_{threshold}',
            'data': paired_df.loc[
                paired_df['total_lulusan'] >= threshold
            ].copy(),
            'rhs': BIN_PRIMARY_RHS,
            'covariance': 'cluster_province',
            'sample_rule': f'total_lulusan >= {threshold}',
        }
        for threshold in [50, 100, 250, 500]
    ],
    {
        'name': 'B_exclude_top_1pct_total',
        'data': paired_df.loc[
            paired_df['total_lulusan'] <= bin_outcome_cutoff
        ].copy(),
        'rhs': BIN_PRIMARY_RHS,
        'covariance': 'cluster_province',
        'sample_rule': 'exclude total graduates above percentile 99',
    },
    {
        'name': 'B_replace_professor_with_senior',
        'data': paired_df,
        'rhs': BIN_PRIMARY_RHS.replace('z_prof_share', 'z_senior_share'),
        'covariance': 'cluster_province',
        'sample_rule': 'senior faculty replaces professor share',
    },
    {
        'name': 'B_without_faculty_block',
        'data': paired_df,
        'rhs': BIN_NO_FACULTY_RHS,
        'covariance': 'cluster_province',
        'sample_rule': 'faculty block removed',
    },
    {
        'name': 'B_province_fixed_effects',
        'data': paired_df,
        'rhs': BIN_PRIMARY_RHS + ' + C(provinsi_pt)',
        'covariance': 'cluster_province',
        'sample_rule': 'province fixed effects',
    },
    {
        'name': 'B_cluster_district',
        'data': paired_df,
        'rhs': BIN_PRIMARY_RHS,
        'covariance': 'cluster_district',
        'sample_rule': 'district-cluster covariance',
    },
    {
        'name': 'B_HC1_covariance',
        'data': paired_df,
        'rhs': BIN_PRIMARY_RHS,
        'covariance': 'HC1',
        'sample_rule': 'HC1 covariance',
    },
]

BIN_CONCEPT_MAP = {
    'z_stem_new_share': 'stem_entry_share_per_sd',
    'z_stem_new_prodi_high_share': 'stem_high_accredited_program_share_per_sd',
    'z_pt_high_share': 'high_accreditation_institution_share_per_sd',
    'z_doctor_stem_share': 'stem_doctor_share_per_sd',
    'z_log_doctor_total': 'log_doctor_total_per_sd',
    'z_prof_share': 'professor_share_per_sd',
    'z_senior_share': 'senior_faculty_share_per_sd',
    'z_cert_share': 'certified_faculty_share_per_sd',
}

bin_results = {}
bin_model_rows = []
bin_coefficient_rows = []

for specification in BIN_SPECIFICATIONS:
    name = specification['name']
    data = specification['data']
    result, _ = fit_grouped_binomial(
        specification['rhs'],
        data,
        covariance=specification['covariance'],
    )
    bin_results[name] = result
    bin_model_rows.append(
        {
            'specification': name,
            'n_rows': len(data),
            'log_likelihood': float(result.llf),
            'aic': float(result.aic),
            'deviance': float(result.deviance),
            'pearson_dispersion': float(
                result.pearson_chi2 / result.df_resid
            ),
            'covariance': str(result.cov_type),
            'sample_rule': specification['sample_rule'],
            'converged': bool(result.converged),
        }
    )
    table = extract_ratio_table(result, name, 'odds_ratio')
    table = table.loc[table['term'].isin(BIN_CONCEPT_MAP)].copy()
    table['concept'] = table['term'].map(BIN_CONCEPT_MAP)
    table['sample_rule'] = specification['sample_rule']
    table['comparable_estimand'] = True
    bin_coefficient_rows.append(table)

bin_specification_grid = pd.DataFrame(bin_model_rows)
bin_coefficients = pd.concat(bin_coefficient_rows, ignore_index=True)
bin_stability = term_stability(
    bin_coefficients,
    ratio_column='odds_ratio',
    comparable_only=True,
)

bin_specification_grid.to_csv(
    OUTPUT_TABLE_DIR / '05_stem_binomial_specification_grid.csv',
    index=False,
)
bin_coefficients.to_csv(
    OUTPUT_TABLE_DIR / '05_stem_binomial_coefficients.csv',
    index=False,
)
bin_stability.to_csv(
    OUTPUT_TABLE_DIR / '05_stem_binomial_term_stability.csv',
    index=False,
)

display(bin_specification_grid)
display(bin_stability)
assert bin_specification_grid['converged'].all()


,specification,n_rows,log_likelihood,aic,deviance,pearson_dispersion,covariance,sample_rule,converged
0,B00_primary,606,"-47,289.523401","94,605.046802","90,461.815810",171.344269,cluster,Notebook 04 paired complete-case,True
1,B_total_graduates_ge_50,588,"-47,230.850862","94,487.701724","90,402.137461",176.619795,cluster,total_lulusan >= 50,True
2,B_total_graduates_ge_100,571,"-47,107.575963","94,241.151927","90,227.960238",181.659475,cluster,total_lulusan >= 100,True
3,B_total_graduates_ge_250,502,"-46,115.402517","92,256.805035","88,576.614050",203.654793,cluster,total_lulusan >= 250,True
4,B_total_graduates_ge_500,428,"-44,747.892657","89,521.785313","86,239.802375",234.004513,cluster,total_lulusan >= 500,True
5,B_exclude_top_1pct_total,599,"-44,065.574429","88,157.148857","84,089.302899",157.433807,cluster,exclude total graduates above percentile 99,True
6,B_replace_professor_with_senior,606,"-47,081.599029","94,189.198059","90,045.967067",171.276521,cluster,senior faculty replaces professor share,True
7,B_without_faculty_block,606,"-47,552.379114","95,126.758228","90,987.527236",172.689912,cluster,faculty block removed,True
8,B_province_fixed_effects,606,"-41,426.908052","82,945.816103","78,736.585111",157.560495,cluster,province fixed effects,True
9,B_cluster_district,606,"-47,289.523401","94,605.046802","90,461.815810",171.344269,cluster,district-cluster covariance,True


,concept,n_specifications,ratio_median,ratio_minimum,ratio_maximum,positive_direction_pct,negative_direction_pct,dominant_direction_pct,significant_pct,ci_excludes_one_pct
0,certified_faculty_share_per_sd,10,0.977777,0.967816,1.013889,10.000000,90.000000,90.000000,0.000000,0.000000
1,high_accreditation_institution_share_per_sd,11,1.000710,0.993742,1.021343,72.727273,27.272727,72.727273,0.000000,0.000000
2,log_doctor_total_per_sd,11,0.963967,0.933179,1.014210,9.090909,90.909091,90.909091,0.000000,0.000000
3,professor_share_per_sd,9,0.946784,0.943020,0.967775,0.000000,100.000000,100.000000,0.000000,0.000000
4,senior_faculty_share_per_sd,1,0.897148,0.897148,0.897148,0.000000,100.000000,100.000000,0.000000,0.000000
5,stem_doctor_share_per_sd,11,1.104404,1.080451,1.116479,100.000000,0.000000,100.000000,18.181818,18.181818
6,stem_entry_share_per_sd,11,3.082263,2.692393,3.113457,100.000000,0.000000,100.000000,100.000000,100.000000
7,stem_high_accredited_program_share_per_sd,11,1.147998,1.072130,1.150260,100.000000,0.000000,100.000000,81.818182,81.818182


In [15]:

# Leave-one-province-out untuk efek komposisi mahasiswa baru STEM.
bin_primary_result = bin_results['B00_primary']
bin_full_coefficient = float(bin_primary_result.params[BIN_ENTRY_TERM])
bin_loo_rows = []

for province in sorted(paired_df['provinsi_pt'].unique()):
    subset = paired_df.loc[paired_df['provinsi_pt'].ne(province)].copy()
    result, _ = fit_grouped_binomial(
        BIN_PRIMARY_RHS,
        subset,
        covariance='cluster_province',
    )
    confidence = result.conf_int().loc[BIN_ENTRY_TERM]
    coefficient = float(result.params[BIN_ENTRY_TERM])
    bin_loo_rows.append(
        {
            'excluded_province': province,
            'coefficient': coefficient,
            'odds_ratio': math.exp(coefficient),
            'odds_ratio_ci_lower': math.exp(float(confidence.iloc[0])),
            'odds_ratio_ci_upper': math.exp(float(confidence.iloc[1])),
            'p_value': float(result.pvalues[BIN_ENTRY_TERM]),
            'absolute_log_or_deviation': abs(
                coefficient - bin_full_coefficient
            ),
        }
    )

bin_loo = pd.DataFrame(bin_loo_rows)
bin_loo.to_csv(
    OUTPUT_TABLE_DIR / '05_stem_binomial_leave_one_province_out.csv',
    index=False,
)

display(
    bin_loo.sort_values('absolute_log_or_deviation', ascending=False).head(10)
)


,excluded_province,coefficient,odds_ratio,odds_ratio_ci_lower,odds_ratio_ci_upper,p_value,absolute_log_or_deviation
3,Prov. Banten,0.997416,2.711266,2.228208,3.299048,0.000000,0.128173
21,Prov. Nusa Tenggara Barat,1.193505,3.298624,2.650728,4.104881,0.000000,0.067917
11,Prov. Jawa Timur,1.089055,2.971464,2.290003,3.855716,0.000000,0.036534
7,Prov. Gorontalo,1.159209,3.187412,2.499565,4.064546,0.000000,0.033621
27,Prov. Sulawesi Selatan,1.108404,3.029521,2.334896,3.930795,0.000000,0.017184
1,Prov. Bali,1.109621,3.033210,2.352494,3.910898,0.000000,0.015967
19,Prov. Maluku,1.140681,3.128898,2.443833,4.006004,0.000000,0.015092
12,Prov. Kalimantan Barat,1.134099,3.108370,2.427257,3.980611,0.000000,0.008510
10,Prov. Jawa Tengah,1.133358,3.106069,2.386618,4.042398,0.000000,0.007769
28,Prov. Sulawesi Tengah,1.133357,3.106066,2.424607,3.979057,0.000000,0.007768


In [16]:

plot_data = bin_coefficients.loc[
    bin_coefficients['concept'].eq('stem_entry_share_per_sd')
].sort_values('odds_ratio')

fig_height = max(6, 0.38 * len(plot_data))
fig, ax = plt.subplots(figsize=(10, fig_height))
positions = np.arange(len(plot_data))
errors = np.vstack(
    [
        plot_data['odds_ratio'] - plot_data['odds_ratio_ci_lower'],
        plot_data['odds_ratio_ci_upper'] - plot_data['odds_ratio'],
    ]
)
ax.errorbar(
    plot_data['odds_ratio'],
    positions,
    xerr=errors,
    fmt='o',
    capsize=3,
)
ax.axvline(1, linestyle='--', linewidth=1)
ax.set_yticks(positions)
ax.set_yticklabels(plot_data['model'])
ax.set_xscale('log')
ax.set_xlabel('Odds Ratio (skala log)')
ax.set_title('Efek Komposisi Mahasiswa Baru STEM pada Spesifikasi Alternatif')
ax.grid(axis='x', alpha=0.25)
save_figure(fig, '05_07_stem_entry_composition_or.png')


PosixPath('/mnt/data/work_project05/pt_education_research_project/outputs/figures/05_07_stem_entry_composition_or.png')


## 7. Sensitivitas model aliran lulusan STEM

Model menghitung jumlah lulusan relatif terhadap mahasiswa baru pada tahun yang
sama. Rasio ini adalah indikator aliran sistem, bukan completion rate cohort.


In [17]:

FLOW_STEM_TERM = 'C(a_stem, Treatment(reference="NON STEM"))[T.STEM]'
FLOW_INTERACTION_PROGRAM_FORMULA = (
    'C(a_stem, Treatment(reference="NON STEM")):z_new_prodi_high_share'
)
FLOW_INTERACTION_DOCTOR_FORMULA = (
    'C(a_stem, Treatment(reference="NON STEM")):z_log_doctor_aligned'
)
FLOW_INTERACTION_PROGRAM = (
    'C(a_stem, Treatment(reference="NON STEM"))[T.STEM]:z_new_prodi_high_share'
)
FLOW_INTERACTION_DOCTOR = (
    'C(a_stem, Treatment(reference="NON STEM"))[T.STEM]:z_log_doctor_aligned'
)

FLOW_BASE = (
    'C(a_stem, Treatment(reference="NON STEM"))'
    ' + C(nm_jenj_didik, Treatment(reference="S1"))'
)
FLOW_MAIN_BLOCK = (
    'z_new_pt_high_share + z_new_prodi_high_share + z_log_pt'
    ' + z_pt_high_share + z_log_doctor_aligned'
    ' + z_prof_share + z_cert_share'
)
FLOW_PRIMARY_FORMULA = (
    'jumlah_lulusan ~ '
    + FLOW_BASE
    + ' + '
    + FLOW_MAIN_BLOCK
    + ' + '
    + FLOW_INTERACTION_PROGRAM_FORMULA
    + ' + '
    + FLOW_INTERACTION_DOCTOR_FORMULA
)
FLOW_NO_INTERACTION_FORMULA = (
    'jumlah_lulusan ~ ' + FLOW_BASE + ' + ' + FLOW_MAIN_BLOCK
)

flow_outcome_cutoff = flow_df['jumlah_lulusan'].quantile(0.99)
FLOW_SPECIFICATIONS = [
    {
        'name': 'F00_primary_interactions',
        'data': flow_df,
        'formula': FLOW_PRIMARY_FORMULA,
        'covariance': 'cluster_province',
        'sample_rule': 'Notebook 04 flow complete-case',
        'interactions': True,
    },
    *[
        {
            'name': f'F_new_students_ge_{threshold}',
            'data': flow_df.loc[
                flow_df['jumlah_mahasiswa_baru'] >= threshold
            ].copy(),
            'formula': FLOW_PRIMARY_FORMULA,
            'covariance': 'cluster_province',
            'sample_rule': f'jumlah_mahasiswa_baru >= {threshold}',
            'interactions': True,
        }
        for threshold in [25, 50, 100, 250, 500]
    ],
    {
        'name': 'F_exclude_top_1pct_outcome',
        'data': flow_df.loc[
            flow_df['jumlah_lulusan'] <= flow_outcome_cutoff
        ].copy(),
        'formula': FLOW_PRIMARY_FORMULA,
        'covariance': 'cluster_province',
        'sample_rule': 'exclude outcome above percentile 99',
        'interactions': True,
    },
    {
        'name': 'F_replace_professor_with_senior',
        'data': flow_df,
        'formula': FLOW_PRIMARY_FORMULA.replace(
            'z_prof_share', 'z_senior_share'
        ),
        'covariance': 'cluster_province',
        'sample_rule': 'senior faculty replaces professor share',
        'interactions': True,
    },
    {
        'name': 'F_without_interactions',
        'data': flow_df,
        'formula': FLOW_NO_INTERACTION_FORMULA,
        'covariance': 'cluster_province',
        'sample_rule': 'STEM interactions removed',
        'interactions': False,
    },
    {
        'name': 'F_province_fixed_effects',
        'data': flow_df,
        'formula': FLOW_PRIMARY_FORMULA + ' + C(provinsi_pt)',
        'covariance': 'cluster_province',
        'sample_rule': 'province fixed effects',
        'interactions': True,
    },
    {
        'name': 'F_cluster_district',
        'data': flow_df,
        'formula': FLOW_PRIMARY_FORMULA,
        'covariance': 'cluster_district',
        'sample_rule': 'district-cluster covariance',
        'interactions': True,
    },
    {
        'name': 'F_HC1_covariance',
        'data': flow_df,
        'formula': FLOW_PRIMARY_FORMULA,
        'covariance': 'HC1',
        'sample_rule': 'HC1 covariance',
        'interactions': True,
    },
]

FLOW_CONCEPT_MAP = {
    FLOW_STEM_TERM: 'stem_vs_nonstem_at_mean_capacity',
    'z_new_pt_high_share': 'new_students_high_accredited_institution_per_sd',
    'z_new_prodi_high_share': 'new_students_high_accredited_program_per_sd',
    'z_log_pt': 'log_number_of_institutions_per_sd',
    'z_pt_high_share': 'high_accreditation_institution_share_per_sd',
    'z_log_doctor_aligned': 'aligned_doctor_capacity_per_sd',
    'z_prof_share': 'professor_share_per_sd',
    'z_senior_share': 'senior_faculty_share_per_sd',
    'z_cert_share': 'certified_faculty_share_per_sd',
    FLOW_INTERACTION_PROGRAM: 'stem_x_high_accredited_program',
    FLOW_INTERACTION_DOCTOR: 'stem_x_aligned_doctor_capacity',
}

flow_results = {}
flow_model_rows = []
flow_coefficient_rows = []
flow_combined_rows = []

for specification in FLOW_SPECIFICATIONS:
    name = specification['name']
    data = specification['data']
    result = fit_nb2(
        specification['formula'],
        data,
        offset_column='jumlah_mahasiswa_baru',
        covariance=specification['covariance'],
    )
    flow_results[name] = result
    metadata = {
        'sample_rule': specification['sample_rule'],
        'offset_column': 'jumlah_mahasiswa_baru',
        'estimand': 'graduate-entry flow intensity',
        'comparable_estimand': True,
        'contains_interactions': specification['interactions'],
    }
    flow_model_rows.append(model_row(name, result, len(data), metadata))

    table = extract_ratio_table(result, name, 'irr')
    table = table.loc[table['term'].isin(FLOW_CONCEPT_MAP)].copy()
    table['concept'] = table['term'].map(FLOW_CONCEPT_MAP)
    for key, value in metadata.items():
        table[key] = value
    flow_coefficient_rows.append(table)

    # STEM vs non-STEM combined effects at three capacity profiles.
    for profile_name, program_z, doctor_z in [
        ('low_capacity_minus_1sd', -1.0, -1.0),
        ('mean_capacity', 0.0, 0.0),
        ('high_capacity_plus_1sd', 1.0, 1.0),
    ]:
        weights = {FLOW_STEM_TERM: 1.0}
        if specification['interactions']:
            weights[FLOW_INTERACTION_PROGRAM] = program_z
            weights[FLOW_INTERACTION_DOCTOR] = doctor_z
        combined = linear_combination_ratio(
            result,
            weights,
            label=profile_name,
        )
        combined['specification'] = name
        combined['sample_rule'] = specification['sample_rule']
        flow_combined_rows.append(combined)

flow_specification_grid = pd.DataFrame(flow_model_rows)
flow_coefficients = pd.concat(flow_coefficient_rows, ignore_index=True)
flow_combined_effects = pd.DataFrame(flow_combined_rows)
flow_stability = term_stability(
    flow_coefficients,
    ratio_column='irr',
    comparable_only=True,
)

flow_specification_grid.to_csv(
    OUTPUT_TABLE_DIR / '05_stem_flow_specification_grid.csv',
    index=False,
)
flow_coefficients.to_csv(
    OUTPUT_TABLE_DIR / '05_stem_flow_coefficients.csv',
    index=False,
)
flow_combined_effects.to_csv(
    OUTPUT_TABLE_DIR / '05_stem_flow_combined_effects.csv',
    index=False,
)
flow_stability.to_csv(
    OUTPUT_TABLE_DIR / '05_stem_flow_term_stability.csv',
    index=False,
)

display(flow_specification_grid)
display(flow_combined_effects.loc[flow_combined_effects['profile'].eq('mean_capacity')])
display(flow_stability)
assert flow_specification_grid['converged'].all()


,specification,n_rows,log_likelihood,aic,bic,df_model,converged,covariance,alpha,sample_rule,offset_column,estimand,comparable_estimand,contains_interactions
0,F00_primary_interactions,1698,"-11,704.898231","23,449.796461","23,558.540589",18.000000,True,cluster,0.779912,Notebook 04 flow complete-case,jumlah_mahasiswa_baru,graduate-entry flow intensity,True,True
1,F_new_students_ge_25,1502,"-10,541.182037","21,122.364075","21,228.655132",18.000000,True,cluster,0.543082,jumlah_mahasiswa_baru >= 25,jumlah_mahasiswa_baru,graduate-entry flow intensity,True,True
2,F_new_students_ge_50,1380,"-9,950.738903","19,941.477806","20,046.074581",18.000000,True,cluster,0.528264,jumlah_mahasiswa_baru >= 50,jumlah_mahasiswa_baru,graduate-entry flow intensity,True,True
3,F_new_students_ge_100,1184,"-8,819.457955","17,678.915909","17,780.448986",18.000000,True,cluster,0.469613,jumlah_mahasiswa_baru >= 100,jumlah_mahasiswa_baru,graduate-entry flow intensity,True,True
4,F_new_students_ge_250,871,"-6,792.584031","13,621.168061","13,707.021617",16.000000,True,cluster,0.365420,jumlah_mahasiswa_baru >= 250,jumlah_mahasiswa_baru,graduate-entry flow intensity,True,True
5,F_new_students_ge_500,602,"-4,912.124068","9,860.248136","9,939.452770",16.000000,True,cluster,0.301469,jumlah_mahasiswa_baru >= 500,jumlah_mahasiswa_baru,graduate-entry flow intensity,True,True
6,F_exclude_top_1pct_outcome,1681,"-11,508.915710","23,057.831419","23,166.374302",18.000000,True,cluster,0.784400,exclude outcome above percentile 99,jumlah_mahasiswa_baru,graduate-entry flow intensity,True,True
7,F_replace_professor_with_senior,1698,"-11,707.976629","23,455.953259","23,564.697386",18.000000,True,cluster,0.782437,senior faculty replaces professor share,jumlah_mahasiswa_baru,graduate-entry flow intensity,True,True
8,F_without_interactions,1698,"-11,715.824774","23,467.649549","23,565.519263",16.000000,True,cluster,0.788810,STEM interactions removed,jumlah_mahasiswa_baru,graduate-entry flow intensity,True,False
9,F_province_fixed_effects,1698,"-11,677.271503","23,460.543006","23,748.714944",51.000000,True,cluster,0.759745,province fixed effects,jumlah_mahasiswa_baru,graduate-entry flow intensity,True,True


,profile,log_ratio,std_error,ratio,ratio_ci_lower,ratio_ci_upper,p_value,specification,sample_rule
1,mean_capacity,-0.359906,0.054488,0.697742,0.627067,0.776382,0.000000,F00_primary_interactions,Notebook 04 flow complete-case
4,mean_capacity,-0.391640,0.036254,0.675947,0.629584,0.725726,0.000000,F_new_students_ge_25,jumlah_mahasiswa_baru >= 25
7,mean_capacity,-0.421914,0.040525,0.655790,0.605717,0.710003,0.000000,F_new_students_ge_50,jumlah_mahasiswa_baru >= 50
10,mean_capacity,-0.398425,0.044862,0.671377,0.614864,0.733083,0.000000,F_new_students_ge_100,jumlah_mahasiswa_baru >= 100
13,mean_capacity,-0.375747,0.048220,0.686776,0.624842,0.754850,0.000000,F_new_students_ge_250,jumlah_mahasiswa_baru >= 250
16,mean_capacity,-0.320334,0.060572,0.725907,0.644646,0.817410,0.000000,F_new_students_ge_500,jumlah_mahasiswa_baru >= 500
19,mean_capacity,-0.353223,0.054908,0.702420,0.630753,0.782231,0.000000,F_exclude_top_1pct_outcome,exclude outcome above percentile 99
22,mean_capacity,-0.389823,0.056221,0.677177,0.606522,0.756061,0.000000,F_replace_professor_with_senior,senior faculty replaces professor share
25,mean_capacity,-0.362341,0.051314,0.696045,0.629447,0.769690,0.000000,F_without_interactions,STEM interactions removed
28,mean_capacity,-0.377419,0.057100,0.685629,0.613035,0.766818,0.000000,F_province_fixed_effects,province fixed effects


,concept,n_specifications,ratio_median,ratio_minimum,ratio_maximum,positive_direction_pct,negative_direction_pct,dominant_direction_pct,significant_pct,ci_excludes_one_pct
0,aligned_doctor_capacity_per_sd,12,1.160368,1.000261,1.218206,100.000000,0.000000,100.000000,25.000000,25.000000
1,certified_faculty_share_per_sd,12,1.067093,0.989054,1.074644,91.666667,8.333333,91.666667,58.333333,58.333333
2,high_accreditation_institution_share_per_sd,12,0.885566,0.870039,0.948865,0.000000,100.000000,100.000000,100.000000,100.000000
3,log_number_of_institutions_per_sd,12,0.885333,0.863282,1.156525,41.666667,58.333333,58.333333,66.666667,66.666667
4,new_students_high_accredited_institution_per_sd,12,1.208849,1.121557,1.240775,100.000000,0.000000,100.000000,41.666667,41.666667
5,new_students_high_accredited_program_per_sd,12,0.868493,0.799498,1.163963,16.666667,83.333333,83.333333,58.333333,58.333333
6,professor_share_per_sd,11,0.935799,0.906658,1.069565,45.454545,54.545455,54.545455,36.363636,36.363636
7,senior_faculty_share_per_sd,1,0.983068,0.983068,0.983068,0.000000,100.000000,100.000000,0.000000,0.000000
8,stem_vs_nonstem_at_mean_capacity,12,0.691411,0.655790,0.725907,0.000000,100.000000,100.000000,100.000000,100.000000
9,stem_x_aligned_doctor_capacity,11,0.826309,0.755885,0.834250,0.000000,100.000000,100.000000,100.000000,100.000000


In [18]:

# Leave-one-province-out untuk STEM vs non-STEM pada kapasitas rata-rata.
flow_primary_result = flow_results['F00_primary_interactions']
flow_full_coefficient = float(flow_primary_result.params[FLOW_STEM_TERM])
flow_loo_rows = []

for province in sorted(flow_df['provinsi_pt'].unique()):
    subset = flow_df.loc[flow_df['provinsi_pt'].ne(province)].copy()
    result = fit_nb2(
        FLOW_PRIMARY_FORMULA,
        subset,
        offset_column='jumlah_mahasiswa_baru',
        covariance='cluster_province',
    )
    confidence = result.conf_int().loc[FLOW_STEM_TERM]
    coefficient = float(result.params[FLOW_STEM_TERM])
    flow_loo_rows.append(
        {
            'excluded_province': province,
            'coefficient': coefficient,
            'irr_mean_capacity': math.exp(coefficient),
            'irr_ci_lower': math.exp(float(confidence.iloc[0])),
            'irr_ci_upper': math.exp(float(confidence.iloc[1])),
            'p_value': float(result.pvalues[FLOW_STEM_TERM]),
            'absolute_log_irr_deviation': abs(
                coefficient - flow_full_coefficient
            ),
            'alpha': float(result.params['alpha']),
            'converged': bool(
                result.mle_retvals.get('converged', False)
            ),
        }
    )

flow_loo = pd.DataFrame(flow_loo_rows)
flow_loo.to_csv(
    OUTPUT_TABLE_DIR / '05_stem_flow_leave_one_province_out.csv',
    index=False,
)

display(
    flow_loo.sort_values('absolute_log_irr_deviation', ascending=False).head(10)
)
assert flow_loo['converged'].all()


,excluded_province,coefficient,irr_mean_capacity,irr_ci_lower,irr_ci_upper,p_value,absolute_log_irr_deviation,alpha,converged
11,Prov. Jawa Timur,-0.402198,0.668849,0.608148,0.735608,0.000000,0.042292,0.721300,True
25,Prov. Riau,-0.333908,0.716119,0.651946,0.786609,0.000000,0.025998,0.764938,True
31,Prov. Sumatera Barat,-0.377255,0.685741,0.616820,0.762364,0.000000,0.017349,0.786548,True
18,Prov. Lampung,-0.374715,0.687485,0.617337,0.765604,0.000000,0.014809,0.786634,True
22,Prov. Nusa Tenggara Timur,-0.348127,0.706009,0.634214,0.785932,0.000000,0.011779,0.778270,True
26,Prov. Sulawesi Barat,-0.348725,0.705587,0.634545,0.784582,0.000000,0.011181,0.779618,True
13,Prov. Kalimantan Selatan,-0.370166,0.690620,0.618515,0.771130,0.000000,0.010260,0.780140,True
21,Prov. Nusa Tenggara Barat,-0.349922,0.704743,0.633352,0.784182,0.000000,0.009984,0.775928,True
27,Prov. Sulawesi Selatan,-0.369385,0.691159,0.618332,0.772564,0.000000,0.009479,0.782617,True
32,Prov. Sumatera Selatan,-0.350876,0.704071,0.630915,0.785710,0.000000,0.009030,0.787829,True


In [19]:

mean_profile = flow_combined_effects.loc[
    flow_combined_effects['profile'].eq('mean_capacity')
].sort_values('ratio')

fig_height = max(6, 0.38 * len(mean_profile))
fig, ax = plt.subplots(figsize=(10, fig_height))
positions = np.arange(len(mean_profile))
errors = np.vstack(
    [
        mean_profile['ratio'] - mean_profile['ratio_ci_lower'],
        mean_profile['ratio_ci_upper'] - mean_profile['ratio'],
    ]
)
ax.errorbar(
    mean_profile['ratio'],
    positions,
    xerr=errors,
    fmt='o',
    capsize=3,
)
ax.axvline(1, linestyle='--', linewidth=1)
ax.set_yticks(positions)
ax.set_yticklabels(mean_profile['specification'])
ax.set_xscale('log')
ax.set_xlabel('STEM vs non-STEM IRR pada kapasitas rata-rata')
ax.set_title('Sensitivitas Intensitas Aliran Lulusan STEM')
ax.grid(axis='x', alpha=0.25)
save_figure(fig, '05_08_stem_flow_irr_sensitivity.png')

profile_plot = flow_combined_effects.loc[
    flow_combined_effects['specification'].eq('F00_primary_interactions')
]
fig, ax = plt.subplots(figsize=(8, 5.5))
positions = np.arange(len(profile_plot))
errors = np.vstack(
    [
        profile_plot['ratio'] - profile_plot['ratio_ci_lower'],
        profile_plot['ratio_ci_upper'] - profile_plot['ratio'],
    ]
)
ax.errorbar(
    profile_plot['ratio'],
    positions,
    xerr=errors,
    fmt='o',
    capsize=3,
)
ax.axvline(1, linestyle='--', linewidth=1)
ax.set_yticks(positions)
ax.set_yticklabels(profile_plot['profile'])
ax.set_xscale('log')
ax.set_xlabel('STEM vs non-STEM IRR')
ax.set_title('Efek Gabungan STEM menurut Profil Kapasitas Regional')
ax.grid(axis='x', alpha=0.25)
save_figure(fig, '05_09_stem_flow_combined_capacity_profiles.png')


PosixPath('/mnt/data/work_project05/pt_education_research_project/outputs/figures/05_09_stem_flow_combined_capacity_profiles.png')


## 8. Matriks konsistensi bukti

Kesimpulan diberi tingkat ketahanan berdasarkan kombinasi spesifikasi,
leave-one-province-out, bootstrap, dan ranking skenario. Kategori ini merupakan
alat transparansi, bukan uji hipotesis baru.


In [20]:

def stability_grade(direction_pct: float, uncertainty_support_pct: float) -> str:
    minimum_support = min(direction_pct, uncertainty_support_pct)
    if minimum_support >= 90:
        return 'Strong'
    if minimum_support >= 70:
        return 'Moderate'
    return 'Sensitive'

# Gender evidence.
gender_female_spec = gender_stability.loc[
    gender_stability['concept'].eq('female_vs_male')
].iloc[0]
gender_active_spec = gender_stability.loc[
    gender_stability['concept'].eq('active_share_per_sd')
].iloc[0]
gender_female_loo = gender_loo_summary.loc[
    gender_loo_summary['concept'].eq('female_vs_male')
].iloc[0]
gender_active_loo = gender_loo_summary.loc[
    gender_loo_summary['concept'].eq('active_share_per_sd')
].iloc[0]
gender_female_boot = bootstrap_summary.loc[
    bootstrap_summary['concept'].eq('female_vs_male')
].iloc[0]
gender_active_boot = bootstrap_summary.loc[
    bootstrap_summary['concept'].eq('active_share_per_sd')
].iloc[0]

# STEM composition evidence.
bin_entry_spec = bin_stability.loc[
    bin_stability['concept'].eq('stem_entry_share_per_sd')
].iloc[0]
bin_entry_loo_direction = float((bin_loo['odds_ratio'] > 1).mean() * 100)
bin_entry_loo_significant = float((bin_loo['p_value'] < 0.05).mean() * 100)

# STEM flow evidence.
flow_stem_spec = flow_stability.loc[
    flow_stability['concept'].eq('stem_vs_nonstem_at_mean_capacity')
].iloc[0]
flow_loo_direction = float((flow_loo['irr_mean_capacity'] < 1).mean() * 100)
flow_loo_significant = float((flow_loo['p_value'] < 0.05).mean() * 100)

best_predictive_model = rank_stability.iloc[0]

integrated_evidence = pd.DataFrame(
    [
        {
            'conclusion': 'Female strata have higher graduate output intensity than male strata',
            'expected_direction': 'ratio > 1',
            'specification_direction_pct': gender_female_spec['positive_direction_pct'],
            'loo_direction_pct': gender_female_loo['positive_direction_pct'],
            'uncertainty_support_pct': min(
                gender_female_spec['ci_excludes_one_pct'],
                gender_female_loo['significant_pct'],
                gender_female_boot['positive_direction_pct'],
            ),
            'robustness_grade': stability_grade(
                min(
                    gender_female_spec['positive_direction_pct'],
                    gender_female_loo['positive_direction_pct'],
                ),
                min(
                    gender_female_spec['ci_excludes_one_pct'],
                    gender_female_loo['significant_pct'],
                    gender_female_boot['positive_direction_pct'],
                ),
            ),
        },
        {
            'conclusion': 'Active-student share is positively associated with graduate output intensity',
            'expected_direction': 'ratio > 1',
            'specification_direction_pct': gender_active_spec['positive_direction_pct'],
            'loo_direction_pct': gender_active_loo['positive_direction_pct'],
            'uncertainty_support_pct': min(
                gender_active_spec['ci_excludes_one_pct'],
                gender_active_loo['significant_pct'],
                gender_active_boot['positive_direction_pct'],
            ),
            'robustness_grade': stability_grade(
                min(
                    gender_active_spec['positive_direction_pct'],
                    gender_active_loo['positive_direction_pct'],
                ),
                min(
                    gender_active_spec['ci_excludes_one_pct'],
                    gender_active_loo['significant_pct'],
                    gender_active_boot['positive_direction_pct'],
                ),
            ),
        },
        {
            'conclusion': 'STEM entry composition predicts STEM graduate composition',
            'expected_direction': 'odds ratio > 1',
            'specification_direction_pct': bin_entry_spec['positive_direction_pct'],
            'loo_direction_pct': bin_entry_loo_direction,
            'uncertainty_support_pct': min(
                bin_entry_spec['ci_excludes_one_pct'],
                bin_entry_loo_significant,
            ),
            'robustness_grade': stability_grade(
                min(
                    bin_entry_spec['positive_direction_pct'],
                    bin_entry_loo_direction,
                ),
                min(
                    bin_entry_spec['ci_excludes_one_pct'],
                    bin_entry_loo_significant,
                ),
            ),
        },
        {
            'conclusion': 'STEM graduate-entry flow intensity is lower than non-STEM at mean capacity',
            'expected_direction': 'IRR < 1',
            'specification_direction_pct': flow_stem_spec['negative_direction_pct'],
            'loo_direction_pct': flow_loo_direction,
            'uncertainty_support_pct': min(
                flow_stem_spec['ci_excludes_one_pct'],
                flow_loo_significant,
            ),
            'robustness_grade': stability_grade(
                min(
                    flow_stem_spec['negative_direction_pct'],
                    flow_loo_direction,
                ),
                min(
                    flow_stem_spec['ci_excludes_one_pct'],
                    flow_loo_significant,
                ),
            ),
        },
        {
            'conclusion': 'One predictive model dominates all province-volume scenarios',
            'expected_direction': 'same model wins every scenario',
            'specification_direction_pct': float(
                best_predictive_model['rmsle_wins']
                / best_predictive_model['scenarios']
                * 100
            ),
            'loo_direction_pct': np.nan,
            'uncertainty_support_pct': float(
                best_predictive_model['rmsle_top_3']
                / best_predictive_model['scenarios']
                * 100
            ),
            'robustness_grade': (
                'Strong'
                if best_predictive_model['rmsle_wins'] == best_predictive_model['scenarios']
                else 'Sensitive'
            ),
        },
    ]
)

integrated_evidence.to_csv(
    OUTPUT_TABLE_DIR / '05_integrated_evidence_matrix.csv',
    index=False,
)
display(integrated_evidence)


,conclusion,expected_direction,specification_direction_pct,loo_direction_pct,uncertainty_support_pct,robustness_grade
0,Female strata have higher graduate output inte...,ratio > 1,100.000000,100.000000,100.000000,Strong
1,Active-student share is positively associated ...,ratio > 1,100.000000,100.000000,100.000000,Strong
2,STEM entry composition predicts STEM graduate ...,odds ratio > 1,100.000000,100.000000,100.000000,Strong
3,STEM graduate-entry flow intensity is lower th...,IRR < 1,100.000000,100.000000,100.000000,Strong
4,One predictive model dominates all province-vo...,same model wins every scenario,60.000000,NaN,100.000000,Sensitive


In [21]:

# Ringkasan machine-readable dan narasi siap artikel.
primary_gender = gender_coefficients.loc[
    gender_coefficients['model'].eq('G00_primary')
]
primary_female = primary_gender.loc[
    primary_gender['concept'].eq('female_vs_male')
].iloc[0]
primary_active = primary_gender.loc[
    primary_gender['concept'].eq('active_share_per_sd')
].iloc[0]
primary_bin_entry = bin_coefficients.loc[
    bin_coefficients['model'].eq('B00_primary')
    & bin_coefficients['concept'].eq('stem_entry_share_per_sd')
].iloc[0]
primary_flow_mean = flow_combined_effects.loc[
    flow_combined_effects['specification'].eq('F00_primary_interactions')
    & flow_combined_effects['profile'].eq('mean_capacity')
].iloc[0]

summary = {
    'gender_specifications': int(len(gender_specification_grid)),
    'gender_bootstrap_successes': int(bootstrap_successes),
    'gender_loo_provinces': int(gender_loo['excluded_province'].nunique()),
    'female_primary_irr': float(primary_female['irr']),
    'female_specification_positive_pct': float(
        gender_female_spec['positive_direction_pct']
    ),
    'female_loo_positive_pct': float(
        gender_female_loo['positive_direction_pct']
    ),
    'female_bootstrap_ci_lower': float(
        gender_female_boot['bootstrap_ci_lower_2_5']
    ),
    'female_bootstrap_ci_upper': float(
        gender_female_boot['bootstrap_ci_upper_97_5']
    ),
    'active_primary_irr': float(primary_active['irr']),
    'active_specification_positive_pct': float(
        gender_active_spec['positive_direction_pct']
    ),
    'active_loo_positive_pct': float(
        gender_active_loo['positive_direction_pct']
    ),
    'predictive_best_mean_rank_model': str(best_predictive_model['model']),
    'predictive_best_model_wins': int(best_predictive_model['rmsle_wins']),
    'predictive_scenarios': int(best_predictive_model['scenarios']),
    'stem_entry_primary_or': float(primary_bin_entry['odds_ratio']),
    'stem_entry_specification_positive_pct': float(
        bin_entry_spec['positive_direction_pct']
    ),
    'stem_entry_loo_positive_pct': float(bin_entry_loo_direction),
    'stem_flow_primary_mean_capacity_irr': float(primary_flow_mean['ratio']),
    'stem_flow_specification_negative_pct': float(
        flow_stem_spec['negative_direction_pct']
    ),
    'stem_flow_loo_negative_pct': float(flow_loo_direction),
}

with (OUTPUT_TABLE_DIR / '05_summary.json').open('w', encoding='utf-8') as handle:
    json.dump(summary, handle, ensure_ascii=False, indent=2)

key_findings = f"""# Ringkasan Robustness and Sensitivity

- Model gender–jenjang diuji pada **{len(gender_specification_grid)} spesifikasi**,
  **{gender_loo['excluded_province'].nunique()} leave-one-province-out fits**, dan
  **{bootstrap_successes} cluster-bootstrap replicates**.
- IRR perempuan pada model utama adalah **{primary_female['irr']:.3f}**.
  Arah IRR >1 bertahan pada **{gender_female_spec['positive_direction_pct']:.1f}%**
  spesifikasi comparable dan **{gender_female_loo['positive_direction_pct']:.1f}%**
  pengeluaran satu provinsi. Interval bootstrap percentile adalah
  **{gender_female_boot['bootstrap_ci_lower_2_5']:.3f}–{gender_female_boot['bootstrap_ci_upper_97_5']:.3f}**.
- IRR proporsi mahasiswa aktif pada model utama adalah
  **{primary_active['irr']:.3f}**. Arah positif bertahan pada
  **{gender_active_spec['positive_direction_pct']:.1f}%** spesifikasi dan
  **{gender_active_loo['positive_direction_pct']:.1f}%** leave-one-province-out fits.
- Model prediktif dengan rata-rata ranking RMSLE provinsi terbaik adalah
  **{best_predictive_model['model']}**, tetapi hanya menang pada
  **{int(best_predictive_model['rmsle_wins'])}/{int(best_predictive_model['scenarios'])}**
  skenario. Karena itu, tidak ada bukti satu algoritma selalu dominan untuk semua
  ukuran provinsi.
- Odds ratio komposisi mahasiswa baru STEM pada model utama adalah
  **{primary_bin_entry['odds_ratio']:.3f}**. Arah positif bertahan pada
  **{bin_entry_spec['positive_direction_pct']:.1f}%** spesifikasi dan
  **{bin_entry_loo_direction:.1f}%** leave-one-province-out fits.
- IRR aliran lulusan STEM terhadap non-STEM pada kapasitas rata-rata adalah
  **{primary_flow_mean['ratio']:.3f}**. Arah IRR <1 bertahan pada
  **{flow_stem_spec['negative_direction_pct']:.1f}%** spesifikasi dan
  **{flow_loo_direction:.1f}%** leave-one-province-out fits.
- Robustness tidak mengubah batas interpretasi: semua hubungan bersifat ekologis,
  bukan efek kausal atau probabilitas individual.
"""

with (OUTPUT_TABLE_DIR / '05_key_findings.md').open('w', encoding='utf-8') as handle:
    handle.write(key_findings)

print(key_findings)


# Ringkasan Robustness and Sensitivity

- Model gender–jenjang diuji pada **15 spesifikasi**,
  **34 leave-one-province-out fits**, dan
  **100 cluster-bootstrap replicates**.
- IRR perempuan pada model utama adalah **1.094**.
  Arah IRR >1 bertahan pada **100.0%**
  spesifikasi comparable dan **100.0%**
  pengeluaran satu provinsi. Interval bootstrap percentile adalah
  **1.067–1.121**.
- IRR proporsi mahasiswa aktif pada model utama adalah
  **1.194**. Arah positif bertahan pada
  **100.0%** spesifikasi dan
  **100.0%** leave-one-province-out fits.
- Model prediktif dengan rata-rata ranking RMSLE provinsi terbaik adalah
  **TweedieRate**, tetapi hanya menang pada
  **3/5**
  skenario. Karena itu, tidak ada bukti satu algoritma selalu dominan untuk semua
  ukuran provinsi.
- Odds ratio komposisi mahasiswa baru STEM pada model utama adalah
  **3.082**. Arah positif bertahan pada
  **100.0%** spesifikasi dan
  **100.0%** leave-one-province-out fits.
- IRR aliran lulusan STEM terhadap no


## 9. Checklist kelayakan Notebook 05


In [22]:

EXPECTED_TABLES = [
    '05_prerequisite_audit.csv',
    '05_gender_specification_grid.csv',
    '05_gender_specification_coefficients.csv',
    '05_gender_term_stability.csv',
    '05_gender_leave_one_province_out.csv',
    '05_gender_leave_one_province_out_summary.csv',
    '05_gender_influential_provinces.csv',
    '05_gender_cluster_bootstrap_estimates.csv',
    '05_gender_cluster_bootstrap_summary.csv',
    '05_gender_cluster_bootstrap_failures.csv',
    '05_predictive_scenario_metrics.csv',
    '05_predictive_rank_stability.csv',
    '05_predictive_generalization_gap_thresholds.csv',
    '05_stem_binomial_specification_grid.csv',
    '05_stem_binomial_coefficients.csv',
    '05_stem_binomial_term_stability.csv',
    '05_stem_binomial_leave_one_province_out.csv',
    '05_stem_flow_specification_grid.csv',
    '05_stem_flow_coefficients.csv',
    '05_stem_flow_combined_effects.csv',
    '05_stem_flow_term_stability.csv',
    '05_stem_flow_leave_one_province_out.csv',
    '05_integrated_evidence_matrix.csv',
    '05_summary.json',
    '05_key_findings.md',
]
EXPECTED_FIGURES = [
    '05_01_gender_female_irr_across_specifications.png',
    '05_02_gender_active_share_irr_across_specifications.png',
    '05_03_gender_influential_provinces.png',
    '05_04_gender_cluster_bootstrap_irr.png',
    '05_05_predictive_rank_stability.png',
    '05_06_predictive_generalization_gap_thresholds.png',
    '05_07_stem_entry_composition_or.png',
    '05_08_stem_flow_irr_sensitivity.png',
    '05_09_stem_flow_combined_capacity_profiles.png',
]

readiness = pd.DataFrame(
    [
        {
            'check': 'Notebook 02–04 prerequisites complete',
            'status': 'PASS' if prerequisite_audit['exists'].all() else 'FAIL',
            'evidence': f'{int(prerequisite_audit["exists"].sum())}/{len(prerequisite_audit)} resources',
        },
        {
            'check': 'Gender specification grid converged',
            'status': 'PASS' if gender_specification_grid['converged'].all() else 'FAIL',
            'evidence': f'{int(gender_specification_grid["converged"].sum())}/{len(gender_specification_grid)} specifications',
        },
        {
            'check': 'Gender leave-one-province-out complete',
            'status': 'PASS' if gender_loo['excluded_province'].nunique() == gender_df['provinsi_pt'].nunique() else 'FAIL',
            'evidence': f'{gender_loo["excluded_province"].nunique()} provinces',
        },
        {
            'check': 'Province cluster bootstrap adequate',
            'status': 'PASS' if bootstrap_successes >= int(0.9 * N_BOOTSTRAP) else 'FAIL',
            'evidence': f'{bootstrap_successes}/{N_BOOTSTRAP} successful replicates',
        },
        {
            'check': 'Predictive scenarios complete',
            'status': 'PASS' if predictive_scenarios['scenario'].nunique() >= 5 else 'FAIL',
            'evidence': f'{predictive_scenarios["scenario"].nunique()} scenarios',
        },
        {
            'check': 'STEM binomial specifications converged',
            'status': 'PASS' if bin_specification_grid['converged'].all() else 'FAIL',
            'evidence': f'{int(bin_specification_grid["converged"].sum())}/{len(bin_specification_grid)} specifications',
        },
        {
            'check': 'STEM flow specifications converged',
            'status': 'PASS' if flow_specification_grid['converged'].all() else 'FAIL',
            'evidence': f'{int(flow_specification_grid["converged"].sum())}/{len(flow_specification_grid)} specifications',
        },
        {
            'check': 'Tables complete',
            'status': 'PASS' if all((OUTPUT_TABLE_DIR / name).exists() for name in EXPECTED_TABLES) else 'FAIL',
            'evidence': f'{sum((OUTPUT_TABLE_DIR / name).exists() for name in EXPECTED_TABLES)}/{len(EXPECTED_TABLES)} tables',
        },
        {
            'check': 'Figures complete',
            'status': 'PASS' if all((OUTPUT_FIGURE_DIR / name).exists() for name in EXPECTED_FIGURES) else 'FAIL',
            'evidence': f'{sum((OUTPUT_FIGURE_DIR / name).exists() for name in EXPECTED_FIGURES)}/{len(EXPECTED_FIGURES)} figures',
        },
    ]
)

readiness.to_csv(
    OUTPUT_TABLE_DIR / '05_readiness_checklist.csv',
    index=False,
)
display(readiness)
assert readiness['status'].eq('PASS').all(), 'Notebook 05 belum seluruhnya PASS.'

manifest_rows = []
for folder, artifact_type in [
    (OUTPUT_TABLE_DIR, 'table'),
    (OUTPUT_FIGURE_DIR, 'figure'),
    (OUTPUT_MODEL_DIR, 'model'),
]:
    for path in sorted(folder.glob('05_*')):
        if path.is_file():
            manifest_rows.append(
                {
                    'artifact_type': artifact_type,
                    'filename': path.name,
                    'relative_path': str(path.relative_to(PROJECT_ROOT)),
                    'size_bytes': path.stat().st_size,
                }
            )

output_manifest = pd.DataFrame(manifest_rows)
output_manifest.to_csv(
    OUTPUT_TABLE_DIR / '05_output_manifest.csv',
    index=False,
)

print('=' * 80)
print('NOTEBOOK 05 LOLOS.')
print('Seluruh analisis utama telah memiliki robustness and sensitivity evidence.')
print('=' * 80)


,check,status,evidence
0,Notebook 02–04 prerequisites complete,PASS,7/7 resources
1,Gender specification grid converged,PASS,15/15 specifications
2,Gender leave-one-province-out complete,PASS,34 provinces
3,Province cluster bootstrap adequate,PASS,100/100 successful replicates
4,Predictive scenarios complete,PASS,5 scenarios
5,STEM binomial specifications converged,PASS,11/11 specifications
6,STEM flow specifications converged,PASS,12/12 specifications
7,Tables complete,PASS,25/25 tables
8,Figures complete,PASS,9/9 figures


NOTEBOOK 05 LOLOS.
Seluruh analisis utama telah memiliki robustness and sensitivity evidence.
